---

# Multimodal (Tabular + Text) Modelling Pipeline Template

*An end-to-end pipeline for tables that mix numeric and categorical columns with free text, for regression or classification. Text enters as TF-IDF, transformer embeddings, and optionally a stacked fine-tuned NLP model, and an ablation measures what each modality adds. Baseline results go to `pipeline-output/baseline/` and final results to `pipeline-output/final/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Toolkit**](#3)
4. [**Data Preparation**](#4)
5. [**Feature Engineering**](#5)
6. [**Validation Strategy**](#6)
7. [**Baseline: Linear / Logistic Regression**](#7)
8. [**Gradient Boosting Models**](#8)
9. [**Hyperparameter Tuning**](#9)
10. [**Ensemble and Model Selection**](#10)
11. [**Final Evaluation on Holdout**](#11)
12. [**Model Interpretation**](#12)
13. [**Error Analysis**](#13)
14. [**Predictions and Export**](#14)


---

# Introduction <a name="1"></a>

---

## Overview

*Competition time is short, so the modelling workflow has to be reliable before the data is even seen. Many competition tables carry text next to structured columns (product descriptions, reviews, complaints, job ads). This notebook turns such a file into a validated model that uses both modalities, with diagnostics and ready-to-submit predictions saved to disk.*

## Aim

*The notebook encodes every text column (TF-IDF summary, transformer embeddings, or both), can stack the out-of-fold predictions of the NLP pipeline, and builds a transparent linear baseline (Linear Regression or Logistic Regression) that always runs with scikit-learn alone, then trains any combination of LightGBM, XGBoost, and CatBoost under the same validation folds, optionally tunes one of them with Optuna, blends them, and reports an unbiased score on a holdout set that no model or decision has touched.*

## How to Configure

*Everything is controlled from the `Settings` class in section 2. Nothing else needs editing.*

| Setting | Purpose | Example |
|---|---|---|
| `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` | Training file per environment (auto-detected) | `'/kaggle/input/comp/train.csv'` |
| `TEST_*_PATH` | Optional unlabeled test file, used to write `submission.csv` | `'.../test.csv'` or `None` |
| `SUBMISSION_FORMAT` | `'label'`, `'proba'` (positive-class probability, or one column per class), or `'auto'` (probabilities for ROC AUC, PR AUC, and log loss) | `'auto'` |
| `EDA_DIR` | Folder with `eda_decisions.json` from the multimodal EDA; every `'auto'` setting below reads it | `'eda-output'` |
| `TARGET`, `TASK` | Target column, and `'auto'`, `'regression'`, or `'classification'` | `'price'`, `'auto'` |
| `DROP_COLS`, `SENTINELS`, `DROP_WEAK` | Columns to drop, placeholder values to blank, weak features (`'auto'` from EDA) | `['const_col']`, `{'key': [-1]}` |
| `ID_COLS`, `DROP_COLS` | Columns never used as features | `['id']` |
| `CV_SCHEME`, `CV_GROUP`, `TIME_COL` | Validation design: `'auto'`, `'kfold'`, `'stratified'`, `'group'`, `'time'` | `'group'`, `'user_id'` |
| `METRIC` | Selection metric, or `'auto'` (RMSE, ROC AUC for binary, macro F1 for multiclass) | `'mae'`, `'f1_macro'` |
| `MODELS` | Any subset of `['lightgbm', 'xgboost', 'catboost']`, or `[]` for baseline only | `['catboost']` |
| `TEXT_MODE` | Text columns as features: `'tfidf'` (TF-IDF to SVD), `'embed'` (transformer embeddings to PCA), `'both'`, `'none'`, or `'auto'` | `'both'` |
| `EMBED_MODEL`, `EMBED_DIM` | Hugging Face encoder for the embeddings and its PCA size | `'indobenchmark/indobert-base-p1'`, `32` |
| `NLP_OOF_DIR` | `pipeline-output/final` folder of the NLP pipeline; its out-of-fold predictions become stacking features | `'../nlp/pipeline-output/final'` |
| `MODALITY_ABLATION` | Score tabular-only vs each text feature family, to show what each modality adds | `True` |
| `TUNE`, `TUNE_MODEL`, `N_TRIALS` | Optional Optuna search for one model | `True`, `'lightgbm'`, `40` |

*A model whose library is not installed is skipped with a warning, so the notebook never breaks on a missing dependency. Competition-specific features go into the `custom_features` function in section 5.*

## Metric

***RMSE** is the default regression metric. It penalises large errors quadratically and is reported together with MAE and R-squared, which explains the share of target variance captured.*

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2} \qquad R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

***ROC AUC** is the default for binary classification because it is threshold-free and robust to imbalance. **Macro F1** is the default for multiclass because it weights every class equally.*

$$\text{Macro F1} = \frac{1}{K}\sum_{k=1}^{K} \frac{2 \cdot P_k \cdot R_k}{P_k + R_k}$$

## Dataset

*Any table with a target and at least one free-text column works. Text columns are detected automatically (strings averaging six or more words) or listed in `TEXT_COLS`. The example below is the DAC PRS ITS 2025 final dataset, which pairs audio features with full lyrics.*

*The demo dataset contains 35,230 Spotify tracks with audio features, lyrics, genre, release year, and a 0-100 popularity score. The same `track_id` appears in several genres, so a group split on `track_id` is used to prevent leakage.*

**Attributes**

1.  `track_id`, `track_name`, `artists` : Track identifiers and metadata
2.  `danceability` ... `tempo`, `duration_ms` : Spotify audio features
3.  `track_genre`                     : Genre label (43 levels)
4.  `release_year`                    : Year of release
5.  `lyrics`                          : Full lyrics text

**Target**
 `popularity`       : Popularity score from 0 to 100

## Approach: Validated Baseline-to-Ensemble Pipeline

*The pipeline separates a holdout set first, so the final score is never influenced by model selection. All models share identical cross-validation folds on the remaining development set, which makes their out-of-fold scores directly comparable and allows a fair blend.*

```
Raw table
    |
    +-- Preparation   : roles, target encoding/transform, dedup, holdout split
    +-- Features      : dates, frequency enc, missing flags, custom
    +-- Text modality : length stats | TF-IDF -> SVD | transformer embedding -> PCA (cached)
    +-- NLP stacking  : out-of-fold predictions of the fine-tuned NLP pipeline (optional)
    +-- CV folds      : kfold | stratified | group | time (shared by every model)
    |
    +-- Baseline      : impute -> scale/power -> one-hot / target enc -> Linear | Logistic
    |                    -> pipeline-output/baseline/
    +-- Boosting      : LightGBM | XGBoost | CatBoost (native categoricals, early stopping)
    +-- Tuning        : Optuna on one model (optional)
    +-- Ensemble      : non-negative weight blend on out-of-fold predictions
    +-- Ablation      : tabular only vs + each text family (what each modality adds)
         |
    Best by CV -> holdout evaluation -> interpretation -> error analysis
         -> pipeline-output/final/ (submission, models, report)
```

*Test-set and holdout predictions are the average of the fold models, so no model is ever refitted on data it is later evaluated on.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

The pipeline runs on CPU. A GPU is only used by the boosting libraries when `USE_GPU = True`. Recorded runtime on the demo dataset with all three boosting models and no tuning is a few minutes on a Kaggle CPU session.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn joblib pyarrow openpyxl lightgbm xgboost catboost optuna torch transformers sentencepiece

## Import Libraries

The boosting libraries and Optuna are imported defensively. A failed import marks the library as unavailable instead of stopping the notebook, which keeps the linear baseline guaranteed to run.

In [ ]:
import os
import re
import json
import hashlib
import time
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.optimize import minimize
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, log_loss, mean_absolute_error,
                             mean_squared_error, median_absolute_error, precision_recall_curve,
                             precision_recall_fscore_support, r2_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, KFold, StratifiedGroupKFold,
                                     StratifiedKFold, TimeSeriesSplit, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder, OneHotEncoder, PowerTransformer, StandardScaler
from IPython.display import Markdown, display

AVAILABLE = {}
try:
    from sklearn.preprocessing import TargetEncoder
    AVAILABLE['target_encoder'] = True
except ImportError:
    AVAILABLE['target_encoder'] = False
try:
    import lightgbm as lgb
    AVAILABLE['lightgbm'] = lgb.__version__
except Exception as e:
    AVAILABLE['lightgbm'] = False
    print(f'[warn] lightgbm unavailable: {str(e).strip().splitlines()[0][:120]}')
try:
    import xgboost as xgb
    AVAILABLE['xgboost'] = xgb.__version__
except Exception as e:
    AVAILABLE['xgboost'] = False
    print(f'[warn] xgboost unavailable: {str(e).strip().splitlines()[0][:120]}')
try:
    import catboost
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
    AVAILABLE['catboost'] = catboost.__version__
except Exception as e:
    AVAILABLE['catboost'] = False
    print(f'[warn] catboost unavailable: {str(e).strip().splitlines()[0][:120]}')
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    AVAILABLE['optuna'] = optuna.__version__
except Exception:
    AVAILABLE['optuna'] = False
try:
    import torch
    from transformers import AutoModel, AutoTokenizer
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
    AVAILABLE['transformers'] = True
except Exception:
    AVAILABLE['transformers'] = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
print(f'scikit-learn {sklearn.__version__} | available: {AVAILABLE}')

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

All paths, column roles, validation choices, model choices, and hyperparameters are centralised here. Switching between Kaggle, Colab, and local execution only requires editing the path lines in block 1. Any option set to `None`, `[]`, or `'auto'` is either skipped or inferred from the data.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}

    # 2) Columns
    EDA_DIR       = 'eda-output'
    TARGET        = 'auto'
    TASK          = 'auto'
    ID_COLS       = 'auto'
    DROP_COLS     = 'auto'
    SENTINELS     = 'auto'
    DATE_COLS     = []
    NUM_COLS      = 'auto'
    CAT_COLS      = 'auto'
    TEXT_COLS     = 'auto'
    SUBMISSION_ID = None
    SUBMISSION_FORMAT = 'auto'

    # 3) Validation
    CV_SCHEME    = 'auto'
    CV_GROUP     = 'auto'
    TIME_COL     = 'auto'
    N_FOLDS      = 5
    HOLDOUT_SIZE = 0.2
    DEDUP        = 'auto'
    METRIC       = 'auto'

    # 4) Preprocessing and features
    TARGET_TRANSFORM   = 'auto'
    CLASS_WEIGHT       = 'auto'
    DROP_WEAK          = 'auto'
    DISCRETE_MAX       = 15
    MAX_OHE_LEVELS     = 30
    GBM_MAX_CAT_LEVELS = 200
    ADD_FREQ_ENCODING  = True
    ADD_MISSING_FLAGS  = True
    TEXT_SVD_DIM       = 16

    # 4b) Multimodal: text columns and NLP stacking
    TEXT_MODE          = 'auto'
    EMBED_MODEL        = 'intfloat/multilingual-e5-base'
    EMBED_DIM          = 32
    EMBED_MAX_LENGTH   = 256
    EMBED_BATCH        = 32
    NLP_OOF_DIR        = None
    MODALITY_ABLATION  = True

    # 5) Models
    BASELINE       = 'linear'
    MODELS         = ['lightgbm', 'xgboost', 'catboost']
    N_ESTIMATORS   = 3000
    LEARNING_RATE  = 0.05
    EARLY_STOPPING = 100
    USE_GPU        = False
    MODEL_PARAMS   = {'lightgbm': {}, 'xgboost': {}, 'catboost': {}}
    TUNE           = False
    TUNE_MODEL     = 'lightgbm'
    N_TRIALS       = 30
    TUNE_TIMEOUT   = 900
    TUNE_FOLDS     = 3
    ENSEMBLE       = True

    # 6) Outputs
    DATA_PATH    = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH    = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    OUTPUT_DIR   = Path('/kaggle/working/pipeline-output') if _ON_KAGGLE else Path('pipeline-output')
    BASELINE_DIR = OUTPUT_DIR / 'baseline'
    FINAL_DIR    = OUTPUT_DIR / 'final'

CFG = Settings()
for d in (CFG.BASELINE_DIR / 'figures', CFG.FINAL_DIR / 'figures', CFG.FINAL_DIR / 'models', CFG.FINAL_DIR / 'oof'):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Test path   : {CFG.TEST_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Load Dataset

The path is resolved first. On Kaggle a missing path triggers a listing of `/kaggle/input` and a fallback to the largest tabular file. The reader is chosen from the file extension. Columns are cleaned in section 4, once the EDA decisions are known.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = [f for f in r.rglob('*') if f.is_file() and f.suffix.lower() in TABULAR_EXT
                     and not SKIP_DIRS & set(f.parts)]
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, CFG.TEST_PATH)
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found and no train file was discovered. Edit the path lines in Settings.'
if TEST_PATH is None:
    print('[warn] No test file configured or found in the dataset folder: submission.csv will not be written.')
df_raw = load_table(DATA_PATH, **CFG.READ_KWARGS)
test_raw = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
SAMPLE = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
print(f'Train : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols  <- {DATA_PATH}')
if test_raw is not None:
    print(f'Test  : {test_raw.shape[0]:,} rows x {test_raw.shape[1]} cols  <- {TEST_PATH}')

---

# Toolkit <a name="3"></a>

---

The same visual identity as the EDA template is applied to every chart: a deep navy primary, a coral accent, and eight categorical colours. `save_fig` writes each figure into the `figures/` folder of either the baseline or the final output directory with a running number.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])
CMAP_DIV  = LinearSegmentedColormap.from_list('div', ['#EE6C4D', '#FBE3DC', '#F7F7F7', '#D6E4F0', '#3D5A80'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = {'baseline': 0, 'final': 0}

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, stage='final'):
    FIG_COUNTER[stage] += 1
    root = CFG.BASELINE_DIR if stage == 'baseline' else CFG.FINAL_DIR
    path = root / 'figures' / f'{FIG_COUNTER[stage]:02d}_{slug(name)}.png'
    fig.savefig(path, facecolor='white')
    plt.show()
    plt.close(fig)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def bar_labels(ax, fmt='{:,.0f}', pad=3, size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues],
                         padding=pad, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

def to_json(obj, path):
    path.write_text(json.dumps(obj, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)), encoding='utf-8')

## Metrics

Every model is scored with the same functions. Regression predictions are values on the original target scale. Classification predictions are probability matrices with one column per class, and labels are derived from them with `to_label`, which applies a tuned threshold for binary tasks when one has been chosen.

In [ ]:
GREATER_IS_BETTER = {'rmse': False, 'mae': False, 'medae': False, 'rmsle': False, 'r2': True,
                     'roc_auc': True, 'pr_auc': True, 'f1_macro': True, 'f1_weighted': True,
                     'f1': True, 'accuracy': True, 'balanced_accuracy': True, 'logloss': False}
THRESHOLD = [0.5]

def to_label(proba):
    if proba.shape[1] == 2:
        return (proba[:, 1] >= THRESHOLD[0]).astype(int)
    return proba.argmax(axis=1)

def score_all(y_true, pred):
    if IS_REG:
        out = {'rmse': float(np.sqrt(mean_squared_error(y_true, pred))), 'mae': mean_absolute_error(y_true, pred),
               'medae': median_absolute_error(y_true, pred), 'r2': r2_score(y_true, pred)}
        if y_true.min() >= 0:
            out['rmsle'] = float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.clip(pred, 0, None)))))
        return out
    lab = to_label(pred)
    out = {'accuracy': accuracy_score(y_true, lab), 'balanced_accuracy': balanced_accuracy_score(y_true, lab),
           'f1_macro': f1_score(y_true, lab, average='macro'), 'f1_weighted': f1_score(y_true, lab, average='weighted')}
    p = np.clip(pred, 1e-7, 1)
    out['logloss'] = log_loss(y_true, p / p.sum(axis=1, keepdims=True), labels=list(range(N_CLASSES)))
    try:
        if IS_BINARY:
            out['roc_auc'] = roc_auc_score(y_true, pred[:, 1])
            out['pr_auc'] = average_precision_score(y_true, pred[:, 1])
            out['f1'] = f1_score(y_true, lab)
        else:
            out['roc_auc'] = roc_auc_score(y_true, p / p.sum(axis=1, keepdims=True), multi_class='ovr', labels=list(range(N_CLASSES)))
    except ValueError:
        out['roc_auc'] = np.nan
    return {k: float(v) for k, v in out.items()}

def primary(y_true, pred):
    return score_all(y_true, pred)[METRIC]

def better(a, b):
    return a > b if GREATER_IS_BETTER[METRIC] else a < b

---

# Data Preparation <a name="4"></a>

---

Preparation fixes everything that must be decided before features are built: column roles, the task type, the target encoding or transform, duplicate removal, and the holdout split. Every decision is printed so it can be quoted in the report's analytical steps chapter.

## EDA-Driven Decisions

`eda-output/eda_decisions.json`, written by the tabular EDA notebook, is loaded when it exists. Every setting on `'auto'` takes its value from it, in priority order: an explicit value in `Settings`, then the EDA file, then a value computed from the data. Each decision is recorded with its source and the EDA evidence behind it, and the table is printed at the end of this section and saved to `pipeline-output/final/decisions.csv`.

In [ ]:
eda_path = Path(CFG.EDA_DIR) / 'eda_decisions.json'
EDA = json.loads(eda_path.read_text()) if eda_path.exists() else {}
print(f'EDA decisions: {eda_path.resolve()} ({"found" if EDA else "not found, computing from data"})')
if EDA and Path(EDA.get('source', '')).name != DATA_PATH.name:
    print(f'[warn] EDA was run on {EDA.get("source")}, not {DATA_PATH.name}')
DECISIONS = []

def decide(name, user_value, eda_key, compute, why=''):
    """Resolve a setting: explicit user value > EDA decision > computed fallback. `why` may be a function of the value."""
    if not (isinstance(user_value, str) and user_value == 'auto'):
        value, source = user_value, 'user'
    elif eda_key and EDA.get(eda_key) is not None:
        value, source = EDA[eda_key], 'eda'
    else:
        value, source = compute(), 'computed'
    why = why(value) if callable(why) else why
    DECISIONS.append({'setting': name, 'value': str(value)[:80], 'source': source, 'why': why})
    return value

def guess_target(train, test):
    """Infer the target: the sample-submission target column, else the one train column missing from the test file."""
    if SAMPLE is not None and SAMPLE.shape[1] == 2 and SAMPLE.columns[1] in train.columns:
        return SAMPLE.columns[1]
    if test is not None:
        only_train = [c for c in train.columns if c not in test.columns]
        if len(only_train) == 1:
            return only_train[0]
    return None

T = decide('TARGET', CFG.TARGET, 'target', lambda: guess_target(df_raw, test_raw) or df_raw.columns[-1],
           'target column analysed in the EDA, else sample_submission or the column missing from test')
assert T in df_raw.columns, f'TARGET={T!r} not found. Columns: {list(df_raw.columns)}'
ID_LIST = [c for c in decide('ID_COLS', CFG.ID_COLS, 'id_cols', lambda: [], 'near-unique identifier columns') if c in df_raw.columns]
CV_GROUP = decide('CV_GROUP', CFG.CV_GROUP, 'cv_group', lambda: None,
                  lambda v: 'identifier repeated across rows, so copies must share a fold' if v else 'no repeated identifier found')
CV_GROUP = CV_GROUP if CV_GROUP in df_raw.columns else None
TIME_COL = decide('TIME_COL', CFG.TIME_COL, 'time_col', lambda: None,
                  lambda v: 'time column used when a time split is chosen' if v else 'no time column')
TIME_COL = TIME_COL if TIME_COL in df_raw.columns else None
keep = {T, CV_GROUP, TIME_COL, *ID_LIST}
DROP = [c for c in decide('DROP_COLS', CFG.DROP_COLS, 'drop_cols', lambda: [], 'constant or quasi-constant (one value >= 99%)') if c in df_raw.columns and c not in keep]
SENTINELS = decide('SENTINELS', CFG.SENTINELS, 'sentinels', lambda: {}, 'placeholder values outside the normal range -> missing')
SENTINELS = SENTINELS if isinstance(SENTINELS, dict) else {}

DATE_PATTERN = r'^(\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4})([ T]\d{1,2}:\d{2}(:\d{2})?.*)?$'

def looks_like_date(s):
    """True when at least 90% of a sampled string column matches a date or datetime pattern."""
    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) or pd.api.types.is_bool_dtype(s):
        return False
    st = s.dropna().astype(str).str.strip()
    if st.empty:
        return False
    st = st.sample(min(len(st), 500), random_state=CFG.SEED)
    return st.str.match(DATE_PATTERN).mean() > 0.9

def parse_dates(s):
    year_first = s.dropna().astype(str).str.strip().str.match(r'^\d{4}').mean() > 0.5
    return pd.to_datetime(s, errors='coerce', format='mixed', dayfirst=not year_first)

def prepare(frame):
    frame = frame.drop(columns=[c for c in DROP if c in frame.columns]).copy()
    for c, vals in SENTINELS.items():
        if c in frame.columns and c != T:
            s = frame[c]
            hit = s.isin(vals) if pd.api.types.is_numeric_dtype(s) else s.astype(str).str.strip().str.lower().isin([str(v) for v in vals])
            frame[c] = s.where(~hit)
    for c in frame.columns:
        if c != T and (c in CFG.DATE_COLS or c in EDA.get('datetime_cols', []) or looks_like_date(frame[c])):
            frame[c] = parse_dates(frame[c])
    return frame

df = prepare(df_raw)
test_df = prepare(test_raw) if test_raw is not None else None
n_sent = sum(int(df_raw[c].isin(v).sum()) if pd.api.types.is_numeric_dtype(df_raw[c]) else 0 for c, v in SENTINELS.items() if c in df_raw.columns)
print(f'Target {T!r} | ids {ID_LIST} | dropped {DROP} | sentinel values set to NaN: {n_sent:,} in {list(SENTINELS)}')

## Column Roles

Roles are inferred with the same rules as the EDA template. Numeric columns with at most `DISCRETE_MAX` distinct values are treated as categorical, long multi-word strings as text, and near-unique strings as identifiers. Explicit lists in `Settings` override the inference.

In [ ]:
def infer_roles(frame):
    roles = {}
    for c in frame.columns:
        if c == T:
            continue
        s = frame[c]
        nun = s.nunique(dropna=True)
        if c in ID_LIST:
            roles[c] = 'id'
        elif c in CFG.DATE_COLS or pd.api.types.is_datetime64_any_dtype(s):
            roles[c] = 'datetime'
        elif nun <= 1:
            roles[c] = 'constant'
        elif pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            roles[c] = 'categorical' if nun <= CFG.DISCRETE_MAX else 'numeric'
        elif pd.api.types.is_bool_dtype(s):
            roles[c] = 'categorical'
        else:
            st = s.dropna().astype(str)
            st = st.sample(min(len(st), 3000), random_state=CFG.SEED)
            words = st.str.split().str.len().mean()
            if words >= 6:
                roles[c] = 'text'
            elif nun / max(s.notna().sum(), 1) > 0.95:
                roles[c] = 'id'
            else:
                roles[c] = 'categorical'
    overrides = (('numeric', CFG.NUM_COLS, 'num_cols'), ('categorical', CFG.CAT_COLS, 'cat_cols'), ('text', CFG.TEXT_COLS, 'text_cols'))
    for role, user, key in overrides:
        cols = user if user != 'auto' else EDA.get(key, 'auto')
        if isinstance(cols, (list, tuple)):
            for c in cols:
                if c in roles:
                    roles[c] = role
    return roles

ROLES     = infer_roles(df)
DECISIONS.append({'setting': 'COLUMN_ROLES', 'value': f'{sum(r == "numeric" for r in ROLES.values())} num / {sum(r == "categorical" for r in ROLES.values())} cat / {sum(r == "text" for r in ROLES.values())} text',
                  'source': 'eda' if EDA.get('num_cols') is not None else 'computed', 'why': 'role inference shared with the EDA'})
NUM_COLS  = [c for c, r in ROLES.items() if r == 'numeric']
CAT_COLS  = [c for c, r in ROLES.items() if r == 'categorical']
TEXT_COLS = [c for c, r in ROLES.items() if r == 'text']
DT_COLS   = [c for c, r in ROLES.items() if r == 'datetime']
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for role in ['numeric', 'categorical', 'text', 'datetime', 'id', 'constant']:
    cols = [c for c, r in ROLES.items() if r == role]
    if cols:
        print(f'{role:<12} ({len(cols):>2}): {cols}')

## Target Preparation

Rows with a missing target are dropped. For classification the labels are encoded to `0..K-1`, classes too small to appear in every fold are removed, and class weighting is switched on automatically when the imbalance ratio exceeds 3. For regression a `log1p` transform is applied automatically when the target is non-negative and right-skewed (skew above 1). Models train on the transformed scale and predictions are always mapped back.

In [ ]:
n0 = len(df)
df = df.dropna(subset=[T]).reset_index(drop=True)
print(f'Dropped {n0 - len(df):,} rows with missing target')

TASK = decide('TASK', CFG.TASK, 'task', lambda: 'classification' if (not pd.api.types.is_numeric_dtype(df[T]) or df[T].nunique() <= 20) else 'regression',
              lambda v: 'string or few-valued target' if v == 'classification' else 'numeric target with many distinct values')
IS_REG = TASK == 'regression'
IS_CLF = not IS_REG

if IS_CLF:
    labels = df[T].astype(str)
    vc = labels.value_counts()
    min_count = 2 * CFG.N_FOLDS
    rare = vc[vc < min_count].index.tolist()
    if rare:
        print(f'[warn] dropping classes with fewer than {min_count} rows: {rare}')
        df = df[~labels.isin(rare)].reset_index(drop=True)
        labels = df[T].astype(str)
    LE = LabelEncoder().fit(labels)
    y_all = LE.transform(labels)
    CLASS_NAMES = list(LE.classes_)
    N_CLASSES = len(CLASS_NAMES)
    IS_BINARY = N_CLASSES == 2
    counts = np.bincount(y_all)
    imbalance = counts.max() / counts.min()
    cw = decide('CLASS_WEIGHT', CFG.CLASS_WEIGHT, 'class_weights', lambda: bool(imbalance > 3), f'imbalance ratio {imbalance:.1f}x')
    USE_CW = cw in (True, 'balanced')
    TRANSFORM = None
    print(f'Task: classification | {N_CLASSES} classes | imbalance {imbalance:.1f}x | class weights: {USE_CW}')
else:
    y_all = df[T].astype(float).values
    N_CLASSES, IS_BINARY, USE_CW, CLASS_NAMES = 1, False, False, None
    skew = pd.Series(y_all).skew()
    TRANSFORM = decide('TARGET_TRANSFORM', CFG.TARGET_TRANSFORM, 'target_transform',
                       lambda: 'log1p' if (y_all.min() >= 0 and skew > 1) else 'none', f'target skew {skew:.2f}')
    TRANSFORM = None if TRANSFORM in (None, 'none') else TRANSFORM
    print(f'Task: regression | skew {skew:.2f} | target transform: {TRANSFORM}')

METRIC = decide('METRIC', CFG.METRIC, 'metric', lambda: 'rmse' if IS_REG else 'roc_auc' if IS_BINARY else 'f1_macro',
                'RMSE for regression, ROC AUC for binary, macro F1 for multiclass')
print(f'Selection metric: {METRIC} ({"higher" if GREATER_IS_BETTER[METRIC] else "lower"} is better)')

def fwd(y):
    return np.log1p(y) if TRANSFORM == 'log1p' else y

def inv(y):
    return np.expm1(y) if TRANSFORM == 'log1p' else y

## Duplicate Removal

Exact duplicate rows add no information and inflate cross-validation scores when copies land in different folds, so they are removed. Near-duplicates (same entity, different rows) are handled later by the group split rather than by deletion.

In [ ]:
DEDUP = bool(decide('DEDUP', CFG.DEDUP, 'dedup', lambda: True,
                   lambda v: 'identical rows found in the EDA duplicate audit' if v else 'no identical rows in the EDA duplicate audit'))
if DEDUP:
    key = pd.util.hash_pandas_object(df.astype(str), index=False)
    keep = ~key.duplicated().values
    print(f'Removed {int((~keep).sum()):,} exact duplicate rows ({(~keep).mean():.1%})')
    df = df[keep].reset_index(drop=True)
    y_all = y_all[keep]
print(f'Rows available for modelling: {len(df):,}')

## Holdout Split

The validation scheme follows the EDA: `group` when an identifier repeats across rows (the EDA also measures how much a random split overstates the score), `time` when a time column exists and train and test drift apart, otherwise `stratified` for classification and plain `kfold`. The holdout split follows the same logic. Time order keeps the latest rows, groups never straddle the boundary, and classes keep their proportions. The holdout is used once, in section 11.

In [ ]:
gap = EDA.get('group_vs_random_gap')
why = f'EDA: random vs group K-fold gap {gap:.1%}' if gap is not None else 'repeated ids -> group, drift -> time'
SCHEME = decide('CV_SCHEME', CFG.CV_SCHEME, 'cv_scheme',
                lambda: 'time' if TIME_COL else 'group' if CV_GROUP else ('stratified' if IS_CLF else 'kfold'), why)
if SCHEME == 'time' and TIME_COL is None:
    SCHEME = 'group' if CV_GROUP else 'kfold'
if SCHEME == 'group' and CV_GROUP is None:
    SCHEME = 'stratified' if IS_CLF else 'kfold'
if SCHEME == 'stratified' and IS_REG:
    SCHEME = 'kfold'
GROUPS_ALL = df[CV_GROUP].astype(str).values if SCHEME == 'group' else None

idx = np.arange(len(df))
if CFG.HOLDOUT_SIZE and CFG.HOLDOUT_SIZE > 0:
    if SCHEME == 'time':
        order = np.argsort(df[TIME_COL].values, kind='stable')
        cut = int(len(order) * (1 - CFG.HOLDOUT_SIZE))
        dev_idx, hold_idx = order[:cut], order[cut:]
    elif SCHEME == 'group':
        if IS_CLF:
            sgk = StratifiedGroupKFold(n_splits=max(2, int(round(1 / CFG.HOLDOUT_SIZE))), shuffle=True, random_state=CFG.SEED)
            dev_idx, hold_idx = next(sgk.split(idx, y_all, GROUPS_ALL))
        else:
            gss = GroupShuffleSplit(n_splits=1, test_size=CFG.HOLDOUT_SIZE, random_state=CFG.SEED)
            dev_idx, hold_idx = next(gss.split(idx, y_all, GROUPS_ALL))
    else:
        dev_idx, hold_idx = train_test_split(idx, test_size=CFG.HOLDOUT_SIZE, random_state=CFG.SEED,
                                             stratify=y_all if IS_CLF else None)
else:
    dev_idx, hold_idx = (np.argsort(df[TIME_COL].values, kind='stable') if SCHEME == 'time' else idx), np.array([], dtype=int)

dev_raw, hold_raw = df.iloc[dev_idx].reset_index(drop=True), df.iloc[hold_idx].reset_index(drop=True)
y_dev, y_hold = y_all[dev_idx], y_all[hold_idx]
groups_dev = GROUPS_ALL[dev_idx] if GROUPS_ALL is not None else None
HAS_HOLD = len(hold_idx) > 0
if GROUPS_ALL is not None and HAS_HOLD:
    overlap = len(set(GROUPS_ALL[dev_idx]) & set(GROUPS_ALL[hold_idx]))
    print(f'Group overlap between dev and holdout: {overlap}')
print(f'Scheme: {SCHEME} | dev {len(dev_idx):,} rows | holdout {len(hold_idx):,} rows')
display(pd.DataFrame(DECISIONS).set_index('setting'))

#### Insights

> *Fill in after running.* Record the task, the metric and why it fits the business question, the target transform, how many duplicates and rare classes were removed, and the validation scheme. These decisions belong in the report's methodology chapter.

---

# Feature Engineering <a name="5"></a>

---

Feature engineering here is generic, so it is safe on any dataset. Date columns become calendar parts and elapsed days. Text columns become length statistics plus TF-IDF and/or transformer-embedding summaries (`TEXT_MODE`), and an NLP model can be stacked in. Categorical columns gain a frequency encoding. Numeric columns with gaps gain missing-value flags. Every stateful step is fitted on the development set only and then applied unchanged to the holdout and test sets.

## Text Mode and Embeddings

Text columns are a second modality inside the table. `TEXT_MODE` decides how they enter the models. `'tfidf'` gives a compact TF-IDF to SVD summary, which is fast and captures keywords. `'embed'` encodes each text with a pretrained transformer (`EMBED_MODEL`, mean pooling) and compresses it with PCA fitted on the development set, which captures meaning and works across paraphrases and languages. `'both'` keeps the two families, and `'auto'` picks `'both'` when a GPU is available or the text volume is small, otherwise `'tfidf'`. Embeddings are unsupervised, so computing them on every row does not leak the target, and they are cached on disk so a rerun skips the encoder.

In [ ]:
N_TEXT_ROWS = len(df) + (len(test_df) if test_df is not None else 0)
HAS_GPU = bool(AVAILABLE['transformers'] and torch.cuda.is_available())

def auto_text_mode():
    if not TEXT_COLS:
        return 'none'
    if AVAILABLE['transformers'] and (HAS_GPU or N_TEXT_ROWS * len(TEXT_COLS) <= 30_000):
        return 'both'
    return 'tfidf'

TEXT_MODE = decide('TEXT_MODE', CFG.TEXT_MODE, 'text_mode', auto_text_mode,
                   lambda v: 'no text columns' if not TEXT_COLS else f'{len(TEXT_COLS)} text column(s), {N_TEXT_ROWS:,} rows, GPU: {HAS_GPU}')
if TEXT_MODE in ('embed', 'both') and not AVAILABLE['transformers']:
    print('[warn] torch / transformers are not installed; text falls back to TF-IDF')
    TEXT_MODE = 'tfidf'
USE_TFIDF = bool(TEXT_COLS) and TEXT_MODE in ('tfidf', 'both') and bool(CFG.TEXT_SVD_DIM)
USE_EMBED = bool(TEXT_COLS) and TEXT_MODE in ('embed', 'both')
CACHE_DIR = CFG.OUTPUT_DIR / 'cache'
EMBED_STORE, ENCODER = {}, {}

def load_encoder():
    if not ENCODER:
        device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
        tok = AutoTokenizer.from_pretrained(CFG.EMBED_MODEL)
        model = AutoModel.from_pretrained(CFG.EMBED_MODEL).to(device).eval()
        ENCODER.update(tok=tok, model=model, device=device)
    return ENCODER

def encode(texts):
    """Mean-pooled, L2-normalised sentence embeddings. E5 models get their 'query: ' prefix."""
    enc = load_encoder()
    prefix = 'query: ' if 'e5' in CFG.EMBED_MODEL.lower() else ''
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), CFG.EMBED_BATCH):
            batch = enc['tok']([prefix + t for t in texts[i:i + CFG.EMBED_BATCH]], padding=True, truncation=True,
                               max_length=CFG.EMBED_MAX_LENGTH, return_tensors='pt').to(enc['device'])
            hidden = enc['model'](**batch).last_hidden_state
            mask = batch['attention_mask'].unsqueeze(-1).to(hidden.dtype)
            vec = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            out.append(torch.nn.functional.normalize(vec, dim=1).float().cpu().numpy())
    return np.vstack(out) if out else np.zeros((0, enc['model'].config.hidden_size), dtype=np.float32)

def embed_column(col, texts):
    """Embed each unique text of a column once, cached on disk by model and content."""
    uniq = sorted(set(texts))
    digest = hashlib.md5((CFG.EMBED_MODEL + str(CFG.EMBED_MAX_LENGTH) + '\x00'.join(uniq)).encode('utf-8', 'ignore')).hexdigest()[:16]
    safe = re.sub(r'\W+', '_', col)
    path = CACHE_DIR / f'embed_{safe}_{digest}.npz'
    if path.exists():
        z = np.load(path, allow_pickle=True)
        M, uniq = z['emb'], list(z['texts'])
        print(f'  [{col}] {len(uniq):,} embeddings loaded from cache')
    else:
        t0 = time.time()
        M = encode(uniq)
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, emb=M, texts=np.array(uniq, dtype=object))
        print(f'  [{col}] {len(uniq):,} unique texts embedded with {CFG.EMBED_MODEL} in {time.time() - t0:.0f}s')
    EMBED_STORE[col] = dict(zip(uniq, M))

def embed_lookup(col, s):
    texts = s.fillna('').astype(str).tolist()
    store = EMBED_STORE[col]
    missing = sorted({t for t in texts if t not in store})
    if missing:
        store.update(zip(missing, encode(missing)))
    return np.vstack([store[t] for t in texts]) if texts else np.zeros((0, len(next(iter(store.values())))))

if USE_EMBED:
    for c in TEXT_COLS:
        parts = [df[c]] + ([test_df[c]] if test_df is not None and c in test_df.columns else [])
        embed_column(c, pd.concat(parts).fillna('').astype(str).tolist())
print(f'Text mode: {TEXT_MODE} | TF-IDF: {USE_TFIDF} | embeddings: {USE_EMBED} | text columns: {TEXT_COLS}')

## NLP Stacking Features

When a fine-tuned transformer from the NLP pipeline is available, its predictions are the strongest text signal. `NLP_OOF_DIR` points to that pipeline's `final` folder. Its out-of-fold, holdout, and test predictions are joined on the shared id column and become `nlp_*` features. Every training row must carry an out-of-fold prediction (a model that never saw that row), otherwise the stacking leaks, so the NLP pipeline should run with `TRAIN_FOLDS` covering all folds.

In [ ]:
NLP_FEATS, NLP_KEY = None, None
if CFG.NLP_OOF_DIR:
    nlp_dir = Path(CFG.NLP_OOF_DIR)
    files = ('oof_predictions.csv', 'holdout_predictions.csv', 'test_predictions_full.csv')
    parts = [pd.read_csv(nlp_dir / f) for f in files if (nlp_dir / f).exists()]
    assert parts, f'No NLP prediction files found in {nlp_dir}'
    nlp = pd.concat(parts, ignore_index=True)
    shared = [c for c in nlp.columns if c in df.columns]
    NLP_KEY = next((c for c in shared if c in ID_LIST + [CFG.SUBMISSION_ID]), shared[0] if shared else None)
    assert NLP_KEY, f'The NLP predictions share no id column with the data. NLP columns: {list(nlp.columns)}'
    pred_cols = [c for c in nlp.columns if c.startswith('proba_')]
    pred_cols = pred_cols or [c for c in nlp.columns if c not in (NLP_KEY, 'actual') and pd.api.types.is_numeric_dtype(nlp[c])]
    NLP_FEATS = nlp.drop_duplicates(NLP_KEY).set_index(NLP_KEY)[pred_cols]
    NLP_FEATS.index = NLP_FEATS.index.astype(str)
    NLP_FEATS.columns = ['nlp_' + re.sub(r'\W+', '_', str(c)) for c in pred_cols]
    cover = {name: frame[NLP_KEY].astype(str).isin(NLP_FEATS.index).mean()
             for name, frame in (('train', df), ('test', test_df)) if frame is not None}
    print(f'NLP stacking features {list(NLP_FEATS.columns)} joined on {NLP_KEY!r} | coverage: ' + ', '.join(f'{k} {v:.1%}' for k, v in cover.items()))
    if cover['train'] < 0.95:
        print('[warn] Under 95% of training rows have an NLP prediction. Set TRAIN_FOLDS to every fold in the NLP '
              'pipeline so each row gets an out-of-fold prediction; missing values are left as NaN.')
else:
    print('No NLP stacking features (NLP_OOF_DIR is None).')

## Custom Features Hook

Domain features usually beat generic ones. Add competition-specific features to `custom_features`, which receives the raw rows and the engineered feature frame. It must only use information available in a single row (or fitted on dev), so that holdout and test rows are transformed identically. The commented lines show typical patterns.

In [ ]:
def custom_features(raw, feats):
    # feats['energy_x_dance'] = raw['energy'] * raw['danceability']
    # feats['duration_min'] = raw['duration_ms'] / 60000
    # feats['n_artists'] = raw['artists'].astype(str).str.count(';') + 1
    return feats

## Feature Builder

The builder follows the scikit-learn `fit` / `transform` contract. Categorical values are cast to strings with an explicit `(missing)` level so that every model sees the same categories.

In [ ]:
class FeatureBuilder:
    def fit(self, frame):
        self.num_cols = list(NUM_COLS)
        self.cat_cols = list(CAT_COLS)
        # near-unique columns give a constant frequency in training, which carries no signal
        self.freq = {c: frame[c].astype(str).value_counts(normalize=True) for c in self.cat_cols
                     if frame[c].nunique() <= 0.5 * len(frame)} if CFG.ADD_FREQ_ENCODING else {}
        self.miss_cols = [c for c in self.num_cols if frame[c].isna().any()] if CFG.ADD_MISSING_FLAGS else []
        self.text = {}
        for c in TEXT_COLS if USE_TFIDF else []:
            docs = frame[c].fillna('').astype(str)
            docs = docs.sample(min(len(docs), 30_000), random_state=CFG.SEED)
            vec = TfidfVectorizer(max_features=20_000, min_df=3, sublinear_tf=True, stop_words='english',
                                  token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
            mat = vec.fit_transform(docs)
            k = min(CFG.TEXT_SVD_DIM, mat.shape[1] - 1)
            if k >= 1:
                self.text[c] = (vec, TruncatedSVD(k, random_state=CFG.SEED).fit(mat))
        self.pca = {}
        for c in TEXT_COLS if USE_EMBED else []:
            E = embed_lookup(c, frame[c])
            k = min(CFG.EMBED_DIM, E.shape[1], len(E) - 1)
            if k >= 1:
                self.pca[c] = PCA(k, random_state=CFG.SEED).fit(E)
        return self

    def transform(self, frame):
        out = pd.DataFrame(index=frame.index)
        for c in self.num_cols:
            out[c] = pd.to_numeric(frame[c], errors='coerce') if c in frame.columns else np.nan
        for c in self.cat_cols:
            s = frame[c] if c in frame.columns else pd.Series(np.nan, index=frame.index)
            out[c] = s.astype(object).where(s.notna(), '(missing)').astype(str).astype(object)
        for c in DT_COLS:
            d = pd.to_datetime(frame[c], errors='coerce')
            out[f'{c}_year'], out[f'{c}_month'] = d.dt.year, d.dt.month
            out[f'{c}_dow'], out[f'{c}_doy'] = d.dt.dayofweek, d.dt.dayofyear
            out[f'{c}_days'] = (d - pd.Timestamp('1970-01-01')).dt.days
        for c in TEXT_COLS:
            s = frame[c].fillna('').astype(str) if c in frame.columns else pd.Series('', index=frame.index)
            words = s.str.split()
            out[f'{c}_n_words'] = words.str.len()
            out[f'{c}_n_chars'] = s.str.len()
            out[f'{c}_n_lines'] = s.str.count('\n') + 1
            out[f'{c}_unique_ratio'] = [len(set(w)) / max(len(w), 1) for w in words]
            if c in self.text:
                vec, svd = self.text[c]
                Z = svd.transform(vec.transform(s)) if len(s) else np.zeros((0, svd.n_components))
                for j in range(Z.shape[1]):
                    out[f'{c}_svd{j}'] = Z[:, j]
            if c in self.pca:
                Z = self.pca[c].transform(embed_lookup(c, s)) if len(s) else np.zeros((0, self.pca[c].n_components_))
                for j in range(Z.shape[1]):
                    out[f'{c}_emb{j}'] = Z[:, j]
        if NLP_FEATS is not None:
            key = frame[NLP_KEY].astype(str) if NLP_KEY in frame.columns else pd.Series('', index=frame.index)
            for c in NLP_FEATS.columns:
                out[c] = key.map(NLP_FEATS[c]).values
        for c, f in self.freq.items():
            src = frame[c].astype(str) if c in frame.columns else pd.Series('nan', index=frame.index)
            out[f'{c}_freq'] = src.map(f).fillna(0).astype(float).values
        for c in self.miss_cols:
            out[f'{c}_isna'] = out[c].isna().astype(int)
        if self.miss_cols:
            out['n_missing'] = out[self.miss_cols].isna().sum(axis=1)
        return custom_features(frame, out)

FB = FeatureBuilder().fit(dev_raw)
X_dev = FB.transform(dev_raw).reset_index(drop=True)
X_hold = FB.transform(hold_raw).reset_index(drop=True)
X_test = FB.transform(test_df).reset_index(drop=True) if test_df is not None else None

F_CAT = [c for c in FB.cat_cols if c in X_dev.columns]
F_NUM = [c for c in X_dev.columns if c not in F_CAT]
const = [c for c in F_NUM if X_dev[c].nunique(dropna=False) <= 1] + [c for c in F_CAT if X_dev[c].nunique() <= 1]
if const:
    print(f'Dropping constant features: {const}')
    F_NUM = [c for c in F_NUM if c not in const]
    F_CAT = [c for c in F_CAT if c not in const]
weak = decide('DROP_WEAK', 'auto' if CFG.DROP_WEAK in ('auto', True) else [], 'weak_features', lambda: [],
              'no mutual information and no significant association with the target')
WEAK = [c for c in (weak if isinstance(weak, list) else []) if c in F_NUM + F_CAT]
if WEAK:
    print(f'Dropping weak features flagged by the EDA: {WEAK}')
    F_NUM = [c for c in F_NUM if c not in WEAK]
    F_CAT = [c for c in F_CAT if c not in WEAK]
FEATURES = F_NUM + F_CAT
for frame in (X_dev, X_hold, X_test):
    if frame is not None:
        frame[F_NUM] = frame[F_NUM].astype(float)
print(f'Features: {len(FEATURES)} ({len(F_NUM)} numeric, {len(F_CAT)} categorical)')

The inventory groups the final features by origin, which is the table to quote when the report describes the feature set.

In [ ]:
def origin(c):
    if c in NUM_COLS:
        return 'raw numeric'
    if c in CAT_COLS:
        return 'raw categorical'
    if c.endswith('_freq'):
        return 'frequency encoding'
    if c.endswith('_isna') or c == 'n_missing':
        return 'missing flags'
    if c.startswith('nlp_'):
        return 'NLP model OOF'
    if any(c.startswith(f'{t}_emb') for t in TEXT_COLS):
        return 'text embedding'
    if any(c.startswith(f'{t}_svd') for t in TEXT_COLS):
        return 'text TF-IDF SVD'
    if any(c.startswith(f'{t}_') for t in TEXT_COLS):
        return 'text statistics'
    if any(c.startswith(f'{t}_') for t in DT_COLS):
        return 'date parts'
    return 'custom'

inventory = pd.DataFrame({'feature': FEATURES, 'origin': [origin(c) for c in FEATURES],
                          'type': ['categorical' if c in F_CAT else 'numeric' for c in FEATURES]})
inventory.to_csv(CFG.FINAL_DIR / 'feature_inventory.csv', index=False)
display(inventory.groupby(['origin', 'type']).size().rename('n_features').to_frame())
display(X_dev.head(3))

#### Insights

> *Fill in after running.* Which feature families were created, and which domain features were added in `custom_features`? Justify each custom feature with an EDA finding.

---

# Validation Strategy <a name="6"></a>

---

All models share one fixed list of folds. Group folds keep every row of an entity in the same fold, which prevents near-duplicate leakage. Time folds always validate on rows later than the training rows. Stratified folds preserve class proportions. The diagnostics confirm that folds are balanced in size and target distribution.

In [ ]:
def make_folds(n_splits=None):
    n_splits = n_splits or CFG.N_FOLDS
    idx = np.arange(len(X_dev))
    if SCHEME == 'time':
        return list(TimeSeriesSplit(n_splits=n_splits).split(idx))
    if SCHEME == 'group':
        if IS_CLF:
            return list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED).split(idx, y_dev, groups_dev))
        return list(GroupKFold(n_splits=n_splits).split(idx, y_dev, groups_dev))
    if SCHEME == 'stratified':
        return list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED).split(idx, y_dev))
    return list(KFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED).split(idx))

FOLDS = make_folds()
rows = []
for k, (tr, va) in enumerate(FOLDS):
    r = {'fold': k + 1, 'n_train': len(tr), 'n_valid': len(va)}
    if IS_REG:
        r.update({'valid_mean': y_dev[va].mean(), 'valid_std': y_dev[va].std()})
    else:
        r.update({'valid_classes': len(np.unique(y_dev[va])), 'valid_major_share': np.bincount(y_dev[va]).max() / len(va)})
    if groups_dev is not None:
        r['group_overlap'] = len(set(groups_dev[tr]) & set(groups_dev[va]))
    rows.append(r)
fold_table = pd.DataFrame(rows).set_index('fold')
display(fold_table)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].bar(fold_table.index, fold_table['n_train'], color=SOFT, label='train')
axes[0].bar(fold_table.index, fold_table['n_valid'], bottom=fold_table['n_train'], color=ACCENT, label='valid')
axes[0].set_xlabel('fold')
axes[0].legend()
axes[0].set_title('Rows per fold', loc='left')
if IS_REG:
    data = [y_dev[va] for _, va in FOLDS]
    bp = axes[1].boxplot(data, patch_artist=True, widths=0.5, showfliers=False)
    for patch, col in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    axes[1].set_title(f'{T} distribution in each validation fold', loc='left')
else:
    top = np.argsort(-np.bincount(y_dev))[:8]
    shares = np.array([[np.mean(y_dev[va] == c) for c in top] for _, va in FOLDS])
    bottom = np.zeros(len(FOLDS))
    for j, c in enumerate(top):
        axes[1].bar(range(1, len(FOLDS) + 1), shares[:, j], bottom=bottom, color=PALETTE[j % 10], label=short(CLASS_NAMES[c], 14))
        bottom += shares[:, j]
    axes[1].legend(fontsize=8, ncol=2)
    axes[1].set_title('Class share per validation fold (top 8 classes)', loc='left')
axes[1].set_xlabel('fold')
suptitle(fig, f'Validation design: {SCHEME} {CFG.N_FOLDS}-fold', f'{len(X_dev):,} dev rows, holdout {len(X_hold):,} rows')
save_fig(fig, 'validation_folds', 'final')

#### Insights

> *Fill in after running.* Are the folds balanced, and is the group overlap zero? State why this scheme mirrors how the model will be used (for example, scoring new songs or new users that were never seen in training).

---

# Baseline: Linear / Logistic Regression <a name="7"></a>

---

The baseline is a deliberately simple, fully interpretable linear model that depends only on scikit-learn, so it always runs. Every boosted model must beat it to justify its complexity. It also reveals whether the signal is mostly linear, which is valuable for the report narrative.

## Shared Model Interface

Every model in this notebook is a small wrapper with `fit(X_train, y_train, X_valid, y_valid)` and `predict(X)`. Regression wrappers apply the target transform internally and return values on the original scale. Classification wrappers return a probability matrix over all classes, even when a training fold misses a class. The `cross_validate` function runs any wrapper over the shared folds and averages fold models to predict the holdout and test sets.

In [ ]:
def sample_weights(y):
    if not USE_CW:
        return None
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    w = len(y) / (np.count_nonzero(counts) * np.where(counts == 0, 1, counts))
    return w[y]

def expand_proba(p, classes):
    full = np.zeros((len(p), N_CLASSES))
    full[:, classes] = p
    return full

def cross_validate(name, make, folds=None, verbose=True, eval_sets=True):
    folds = folds or FOLDS
    shape = (len(y_dev),) if IS_REG else (len(y_dev), N_CLASSES)
    oof = np.full(shape, np.nan)
    hold = np.zeros((len(X_hold),) + shape[1:])
    test = np.zeros((len(X_test),) + shape[1:]) if X_test is not None else None
    models, fold_scores, t0 = [], [], time.time()
    for k, (tr, va) in enumerate(folds):
        m = make().fit(X_dev.iloc[tr], y_dev[tr], X_dev.iloc[va], y_dev[va])
        oof[va] = m.predict(X_dev.iloc[va])
        fold_scores.append(primary(y_dev[va], oof[va]))
        if verbose:
            print(f'  [{name}] fold {k + 1}/{len(folds)}  {METRIC} = {fold_scores[-1]:.4f}  ({time.time() - t0:.0f}s)')
        if eval_sets and HAS_HOLD:
            hold += m.predict(X_hold) / len(folds)
        if eval_sets and test is not None:
            test += m.predict(X_test) / len(folds)
        models.append(m)
    covered = ~np.isnan(oof if IS_REG else oof[:, 0])
    cv = score_all(y_dev[covered], oof[covered])
    res = {'name': name, 'oof': oof, 'covered': covered, 'hold': hold, 'test': test, 'models': models,
           'fold_scores': fold_scores, 'cv': cv, 'cv_primary': cv[METRIC], 'cv_std': float(np.std(fold_scores)),
           'time': time.time() - t0}
    res['holdout'] = score_all(y_hold, hold) if HAS_HOLD and eval_sets else {}
    if verbose:
        print(f'  [{name}] CV {METRIC} = {res["cv_primary"]:.4f} +/- {res["cv_std"]:.4f}  in {res["time"]:.0f}s')
    return res

## Linear Preprocessing and Model

Numeric features are median-imputed, standardised, and passed through a Yeo-Johnson power transform, which tames skew and outliers for a linear model. Transformed values are clipped to +/-5 so that a value never seen in training cannot push a prediction to 0 or 1. Low-cardinality categoricals (at most `MAX_OHE_LEVELS` levels) are one-hot encoded with rare levels grouped. High-cardinality categoricals use scikit-learn's cross-fitted `TargetEncoder`, which avoids target leakage, and fall back to their frequency encoding on older scikit-learn versions. The estimator is `LinearRegression` or `LogisticRegression` (multinomial, balanced class weights when imbalance is high). Setting `BASELINE = 'ridge'` swaps in Ridge regression for more stable coefficients.

In [ ]:
class LinearModel:
    def __init__(self, params=None):
        self.params = params or {}

    def _preprocessor(self, X):
        low = [c for c in F_CAT if X[c].nunique() <= CFG.MAX_OHE_LEVELS]
        high = [c for c in F_CAT if c not in low]
        num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                             ('power', PowerTransformer(method='yeo-johnson')),
                             ('clip', FunctionTransformer(np.clip, kw_args={'a_min': -5, 'a_max': 5}, feature_names_out='one-to-one'))])
        steps = [('num', num_pipe, F_NUM)]
        if low:
            steps.append(('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10,
                                                  drop='first', sparse_output=False), low))
        if high and AVAILABLE['target_encoder']:
            te = TargetEncoder(target_type='continuous' if IS_REG else 'auto', random_state=CFG.SEED)
            steps.append(('target_enc', Pipeline([('te', te), ('scale', StandardScaler())]), high))
        return ColumnTransformer(steps, remainder='drop', verbose_feature_names_out=False)

    def fit(self, X, y, X_valid=None, y_valid=None):
        pre = self._preprocessor(X)
        if IS_REG:
            est = Ridge(alpha=self.params.get('alpha', 1.0)) if CFG.BASELINE == 'ridge' else LinearRegression()
            self.pipe = Pipeline([('pre', pre), ('est', est)]).fit(X, fwd(y))
        else:
            self.classes, yl = np.unique(y, return_inverse=True)
            est = LogisticRegression(max_iter=3000, C=self.params.get('C', 1.0),
                                     class_weight='balanced' if USE_CW else None)
            self.pipe = Pipeline([('pre', pre), ('est', est)]).fit(X, yl)
        return self

    def predict(self, X):
        if IS_REG:
            return inv(self.pipe.predict(X))
        return expand_proba(self.pipe.predict_proba(X), self.classes)

    def coefficients(self):
        names = self.pipe.named_steps['pre'].get_feature_names_out()
        coef = self.pipe.named_steps['est'].coef_
        if coef.ndim == 2 and coef.shape[0] > 1:
            return pd.Series(np.abs(coef).mean(axis=0), index=names), True
        return pd.Series(np.ravel(coef), index=names), False

## Baseline Cross-Validation

The baseline runs on the shared folds. A naive predictor (training mean for regression, class priors for classification) is scored alongside it, so the improvement the linear model brings over guessing is explicit.

In [ ]:
def naive_prediction(n):
    if IS_REG:
        return np.full(n, y_dev.mean())
    prior = np.bincount(y_dev, minlength=N_CLASSES) / len(y_dev)
    return np.tile(prior, (n, 1))

BASE = cross_validate('baseline', lambda: LinearModel())
RESULTS = {'baseline': BASE}

NAIVE = {'cv': score_all(y_dev, naive_prediction(len(y_dev)))}
NAIVE['holdout'] = score_all(y_hold, naive_prediction(len(y_hold))) if HAS_HOLD else {}

base_table = pd.DataFrame({'naive (CV)': NAIVE['cv'], 'baseline (CV)': BASE['cv']})
if HAS_HOLD:
    base_table['naive (holdout)'] = NAIVE['holdout']
    base_table['baseline (holdout)'] = BASE['holdout']
display(base_table.round(4))
pd.DataFrame({'fold': range(1, len(FOLDS) + 1), METRIC: BASE['fold_scores']}).to_csv(CFG.BASELINE_DIR / 'cv_scores.csv', index=False)
base_table.to_csv(CFG.BASELINE_DIR / 'metrics.csv')

## Baseline Diagnostics

The diagnostics function below is shared with the final evaluation. For regression it shows predicted against actual values, residuals against predictions, and the residual distribution. For classification it shows the confusion matrix together with ROC and precision-recall curves (binary) or per-class F1 (multiclass).

In [ ]:
def plot_diagnostics(y_true, pred, title, stage, subtitle=''):
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.4))
    if IS_REG:
        res = y_true - pred
        ax = axes[0]
        ax.hexbin(y_true, pred, gridsize=40, cmap=CMAP_SEQ, mincnt=1, bins='log', linewidths=0)
        lims = [min(y_true.min(), pred.min()), max(y_true.max(), pred.max())]
        ax.plot(lims, lims, color=ACCENT, ls='--', lw=1.5)
        ax.set_xlabel(f'actual {T}')
        ax.set_ylabel('predicted')
        ax.set_title('Predicted vs actual', loc='left')
        ax = axes[1]
        ax.hexbin(pred, res, gridsize=40, cmap=CMAP_SEQ, mincnt=1, bins='log', linewidths=0)
        d = pd.DataFrame({'p': pred, 'r': res})
        b = d.groupby(pd.qcut(d['p'].rank(method='first'), 20), observed=True).mean()
        ax.plot(b['p'], b['r'], color=ACCENT, marker='o', ms=3, lw=2)
        ax.axhline(0, color=DARK, lw=1)
        ax.set_xlabel('predicted')
        ax.set_ylabel('residual (actual - predicted)')
        ax.set_title('Residuals vs predicted (orange = binned mean)', loc='left')
        ax = axes[2]
        sns.histplot(res, bins=60, color=PRIMARY, edgecolor='white', ax=ax, kde=True)
        ax.axvline(0, color=ACCENT, ls='--')
        ax.set_title(f'Residual distribution (MAE {np.abs(res).mean():.3f})', loc='left')
    else:
        lab = to_label(pred)
        keep = np.argsort(-np.bincount(y_true, minlength=N_CLASSES))[:15]
        keep = np.sort(keep)
        cm = confusion_matrix(y_true, lab, labels=list(range(N_CLASSES)), normalize='true')[np.ix_(keep, keep)]
        names = [short(CLASS_NAMES[i], 12) for i in keep]
        sns.heatmap(cm, cmap=CMAP_SEQ, vmin=0, vmax=1, annot=len(keep) <= 12, fmt='.2f', ax=axes[0],
                    xticklabels=names, yticklabels=names, cbar_kws={'shrink': 0.7})
        axes[0].set_xlabel('predicted')
        axes[0].set_ylabel('actual')
        extra = f', top {len(keep)} of {N_CLASSES} classes' if N_CLASSES > len(keep) else ''
        axes[0].set_title(f'Row-normalised confusion matrix{extra}', loc='left')
        if IS_BINARY:
            fpr, tpr, _ = roc_curve(y_true, pred[:, 1])
            axes[1].plot(fpr, tpr, color=PRIMARY, lw=2.2)
            axes[1].fill_between(fpr, tpr, color=SOFT, alpha=0.3)
            axes[1].plot([0, 1], [0, 1], color=MUTED, ls='--')
            axes[1].set_xlabel('false positive rate')
            axes[1].set_ylabel('true positive rate')
            axes[1].set_title(f'ROC curve (AUC {roc_auc_score(y_true, pred[:, 1]):.3f})', loc='left')
            prec, rec, _ = precision_recall_curve(y_true, pred[:, 1])
            axes[2].plot(rec, prec, color=ACCENT, lw=2.2)
            axes[2].axhline(y_true.mean(), color=MUTED, ls='--', label='prevalence')
            axes[2].set_xlabel('recall')
            axes[2].set_ylabel('precision')
            axes[2].legend()
            axes[2].set_title(f'Precision-recall (AP {average_precision_score(y_true, pred[:, 1]):.3f})', loc='left')
        else:
            p, r, f, s = precision_recall_fscore_support(y_true, lab, labels=list(range(N_CLASSES)), zero_division=0)
            per = pd.DataFrame({'f1': f, 'support': s}, index=CLASS_NAMES).sort_values('f1')
            show = per if len(per) <= 25 else pd.concat([per.head(12), per.tail(12)])
            axes[1].barh([short(i, 18) for i in show.index], show['f1'], color=[ACCENT if v < per['f1'].median() else PRIMARY for v in show['f1']])
            axes[1].tick_params(axis='y', labelsize=7)
            axes[1].set_title('Per-class F1 (orange = below median)', loc='left')
            axes[2].scatter(per['support'], per['f1'], color=PRIMARY, s=30, alpha=0.7)
            for name, row in per.iterrows():
                axes[2].annotate(short(name, 12), (row['support'], row['f1']), fontsize=7, color=MUTED)
            axes[2].set_xscale('log')
            axes[2].set_xlabel('support (log)')
            axes[2].set_ylabel('F1')
            axes[2].set_title('F1 vs class size', loc='left')
    suptitle(fig, title, subtitle)
    save_fig(fig, title, stage)

plot_diagnostics(y_dev[BASE['covered']], BASE['oof'][BASE['covered']], 'Baseline out-of-fold diagnostics', 'baseline',
                 f'CV {METRIC} = {BASE["cv_primary"]:.4f}')
if HAS_HOLD:
    plot_diagnostics(y_hold, BASE['hold'], 'Baseline holdout diagnostics', 'baseline',
                     f'holdout {METRIC} = {BASE["holdout"][METRIC]:.4f}')

## Baseline Coefficients

A single baseline is refitted on the full development set to read its coefficients. Because numeric inputs are standardised, coefficient magnitudes are comparable across features. For multiclass problems the mean absolute coefficient across classes is shown.

In [ ]:
BASE_FULL = LinearModel().fit(X_dev, y_dev)
coef, is_abs = BASE_FULL.coefficients()
coef_table = coef.rename('coefficient').to_frame().assign(abs=coef.abs()).sort_values('abs', ascending=False)
coef_table.to_csv(CFG.BASELINE_DIR / 'coefficients.csv')

top = coef_table.head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 0.32 * len(top) + 1.8))
colors = [PRIMARY if is_abs or v > 0 else ACCENT for v in top['coefficient']]
ax.barh([short(i, 34) for i in top.index], top['coefficient'], color=colors)
ax.axvline(0, color=DARK, lw=1)
scale = f'log1p({T})' if TRANSFORM else T
sub = 'mean |coefficient| across classes' if is_abs else f'effect of +1 SD on {scale if IS_REG else "log-odds"}, blue = positive, orange = negative'
suptitle(fig, 'Baseline: top 25 coefficients', sub)
save_fig(fig, 'baseline_coefficients', 'baseline')

## Baseline Export

All baseline artefacts are written to `pipeline-output/baseline/`: fold scores, metrics, out-of-fold and holdout predictions, coefficients, the fold models, and a submission file when a test set is configured.

In [ ]:
def prediction_frame(pred, raw=None, include_truth=None):
    out = pd.DataFrame(index=range(len(pred)))
    id_col = CFG.SUBMISSION_ID or next((c for c in ID_LIST if raw is not None and c in raw.columns), None)
    if id_col and raw is not None and id_col in raw.columns:
        out[id_col] = raw[id_col].values
    if include_truth is not None:
        out['actual'] = include_truth if IS_REG else LE.inverse_transform(include_truth)
    if IS_REG:
        out[T] = pred
    else:
        out[T] = LE.inverse_transform(to_label(pred))
        for j, c in enumerate(CLASS_NAMES):
            out[f'proba_{c}'] = pred[:, j]
    return out

def follow_sample(pred, frame):
    """Shape test predictions like sample_submission: same id order, column names, and label or probability format."""
    sid, tcols = SAMPLE.columns[0], list(SAMPLE.columns[1:])
    if sid not in frame.columns or not tcols or len(frame) != len(pred):
        print(f'[warn] sample submission id column {sid!r} is not in the test file; using the default submission layout')
        return None
    names = [str(c) for c in CLASS_NAMES] if not IS_REG else []
    if not IS_REG and len(tcols) > 1 and set(map(str, tcols)) <= set(names):
        vals = pd.DataFrame(pred[:, [names.index(str(c)) for c in tcols]], columns=tcols)
    else:
        ex = SAMPLE[tcols[0]].dropna()
        looks_proba = pd.api.types.is_float_dtype(ex) and not ex.isin([0, 1]).all()
        fmt = CFG.SUBMISSION_FORMAT if CFG.SUBMISSION_FORMAT != 'auto' else 'proba' if looks_proba else 'label'
        if IS_REG:
            v = pred
        elif fmt == 'proba' and N_CLASSES == 2:
            v = pred[:, 1]
        else:
            v = LE.inverse_transform(to_label(pred))
        vals = pd.DataFrame({tcols[0]: v})
    vals.insert(0, sid, frame[sid].values)
    out = SAMPLE[[sid]].merge(vals, on=sid, how='left')
    n_missing = int(out[vals.columns[1:]].isna().any(axis=1).sum())
    if n_missing:
        print(f'[warn] {n_missing:,} sample-submission ids have no prediction')
    return out

def export_result(res, folder):
    cov = res['covered']
    prediction_frame(res['oof'][cov], dev_raw[cov], y_dev[cov]).to_csv(folder / 'oof_predictions.csv', index=False)
    if HAS_HOLD:
        prediction_frame(res['hold'], hold_raw, y_hold).to_csv(folder / 'holdout_predictions.csv', index=False)
    if res['test'] is not None:
        sub = prediction_frame(res['test'], test_df)
        id_cols = [c for c in sub.columns if c in (ID_LIST + [CFG.SUBMISSION_ID])]
        sub_out = follow_sample(res['test'], test_df) if SAMPLE is not None else None
        if sub_out is None:
            fmt = CFG.SUBMISSION_FORMAT
            if fmt == 'auto':
                fmt = 'proba' if not IS_REG and METRIC in ('roc_auc', 'pr_auc', 'logloss') else 'label'
            if fmt == 'proba' and not IS_REG and N_CLASSES == 2:
                sub_out = sub[id_cols].assign(**{T: res['test'][:, 1]})
            elif fmt == 'proba' and not IS_REG:
                sub_out = sub[id_cols + [f'proba_{c}' for c in CLASS_NAMES]].rename(columns=lambda c: c.removeprefix('proba_'))
            else:
                sub_out = sub[id_cols + [T]]
        sub_out.to_csv(folder / 'submission.csv', index=False)
        sub.to_csv(folder / 'test_predictions_full.csv', index=False)

export_result(BASE, CFG.BASELINE_DIR)
joblib.dump(BASE['models'], CFG.BASELINE_DIR / 'baseline_fold_models.joblib')
to_json({'task': TASK, 'metric': METRIC, 'scheme': SCHEME, 'target_transform': TRANSFORM,
         'model': 'Ridge' if (IS_REG and CFG.BASELINE == 'ridge') else 'LinearRegression' if IS_REG else 'LogisticRegression',
         'cv': BASE['cv'], 'cv_fold_scores': BASE['fold_scores'], 'holdout': BASE['holdout'], 'naive': NAIVE},
        CFG.BASELINE_DIR / 'metrics.json')
print('Baseline files:', sorted(p.name for p in CFG.BASELINE_DIR.iterdir()))

#### Insights

> *Fill in after running.* How much does the linear baseline improve on the naive predictor? Which coefficients dominate, and do their signs match the EDA? A small gain over naive means the signal is weak or non-linear, which motivates the boosted models below.

---

# Gradient Boosting Models <a name="8"></a>

---

Gradient boosted trees are the strongest general-purpose models for tabular data. They capture non-linearities and interactions without manual feature crosses and are insensitive to monotone transforms, so they use the raw engineered features without scaling. Each library handles categoricals natively. LightGBM and XGBoost receive pandas categoricals capped at `GBM_MAX_CAT_LEVELS` levels. XGBoost uses one-vs-rest category splits, because its partition splits overfit badly on high-cardinality columns in small data. CatBoost receives raw strings and applies ordered target statistics. Early stopping on the validation fold sets the number of trees. CatBoost is the slowest of the three on many-class problems, so for more than 10 classes it switches to cheaper categorical combinations automatically, and `USE_GPU = True` speeds it up considerably on Kaggle.

## Categorical Handling

Category levels are fixed from the development set, so every fold, the holdout, and the test set share an identical category mapping. Levels outside the most frequent `GBM_MAX_CAT_LEVELS - 1` are pooled into `(other)`, which also absorbs levels unseen at training time.

In [ ]:
CAT_LEVELS = {}
for c in F_CAT:
    keep = X_dev[c].value_counts().index[:CFG.GBM_MAX_CAT_LEVELS - 1].tolist()
    CAT_LEVELS[c] = keep + ([] if '(other)' in keep else ['(other)'])

def tree_frame(X, kind):
    Z = X[FEATURES].copy()
    for c in F_CAT:
        if kind == 'catboost':
            Z[c] = Z[c].astype(str).astype(object)
        else:
            s = Z[c].where(Z[c].isin(CAT_LEVELS[c]), '(other)')
            Z[c] = pd.Categorical(s, categories=CAT_LEVELS[c])
    return Z

print({c: len(v) for c, v in CAT_LEVELS.items()})

## Model Wrappers

The base class handles everything shared: the target transform for regression, mapping each fold's classes to a contiguous range (so a fold that misses a rare class does not crash), class weights, and expanding probabilities back to all classes. Each subclass only defines its default hyperparameters, the library-specific fit call, gain importance, and exact tree SHAP contributions through the library's own API (no `shap` package required). Defaults can be overridden per model through `Settings.MODEL_PARAMS`.

In [ ]:
class GBMModel:
    kind = None

    def __init__(self, params=None):
        self.params = {**self.defaults(), **CFG.MODEL_PARAMS.get(self.kind, {}), **(params or {})}

    def fit(self, X, y, X_valid=None, y_valid=None):
        Xt = tree_frame(X, self.kind)
        Xv = tree_frame(X_valid, self.kind) if X_valid is not None else None
        w = None
        if IS_REG:
            yt = fwd(y)
            yv = fwd(y_valid) if y_valid is not None else None
        else:
            self.classes = np.unique(y)
            yt = np.searchsorted(self.classes, y)
            w = sample_weights(y)
            yv = None
            if y_valid is not None:
                mask = np.isin(y_valid, self.classes)
                Xv, yv = Xv[mask], np.searchsorted(self.classes, y_valid[mask])
        self._fit(Xt, yt, Xv, yv, w)
        return self

    def predict(self, X):
        Z = tree_frame(X, self.kind)
        if IS_REG:
            return inv(np.asarray(self.model.predict(Z), dtype=float))
        return expand_proba(self.model.predict_proba(Z), self.classes)

    def n_local_classes(self):
        return 1 if IS_REG else len(self.classes)


class LGBModel(GBMModel):
    kind = 'lightgbm'

    def defaults(self):
        p = dict(n_estimators=CFG.N_ESTIMATORS, learning_rate=CFG.LEARNING_RATE, num_leaves=63,
                 min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                 reg_lambda=1.0, cat_smooth=10, random_state=CFG.SEED, n_jobs=-1, verbose=-1)
        if CFG.USE_GPU:
            p['device'] = 'gpu'
        return p

    def _fit(self, X, y, Xv, yv, w):
        cls = lgb.LGBMRegressor if IS_REG else lgb.LGBMClassifier
        self.model = cls(**self.params)
        kw = {}
        if Xv is not None:
            kw = dict(eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING, verbose=False)])
        self.model.fit(X, y, sample_weight=w, **kw)

    def best_iteration(self):
        return self.model.best_iteration_ or self.params['n_estimators']

    def importance(self):
        return pd.Series(self.model.booster_.feature_importance('gain'), index=FEATURES)

    def contributions(self, X):
        c = np.asarray(self.model.predict(tree_frame(X, self.kind), pred_contrib=True))
        k = self.n_local_classes()
        return c.reshape(len(X), k, -1)[:, :, :-1] if k > 2 else c[:, :-1]


class XGBModel(GBMModel):
    kind = 'xgboost'

    def defaults(self):
        p = dict(n_estimators=CFG.N_ESTIMATORS, learning_rate=CFG.LEARNING_RATE, max_depth=6,
                 min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                 tree_method='hist', enable_categorical=True, max_cat_to_onehot=CFG.GBM_MAX_CAT_LEVELS,
                 early_stopping_rounds=CFG.EARLY_STOPPING, random_state=CFG.SEED, n_jobs=-1)
        if CFG.USE_GPU:
            p['device'] = 'cuda'
        return p

    def _fit(self, X, y, Xv, yv, w):
        params = dict(self.params)
        if Xv is None:
            params.pop('early_stopping_rounds', None)
        cls = xgb.XGBRegressor if IS_REG else xgb.XGBClassifier
        self.model = cls(**params)
        self.model.fit(X, y, sample_weight=w, eval_set=[(Xv, yv)] if Xv is not None else None, verbose=False)

    def best_iteration(self):
        return getattr(self.model, 'best_iteration', None) or self.params['n_estimators']

    def importance(self):
        score = self.model.get_booster().get_score(importance_type='total_gain')
        return pd.Series(score).reindex(FEATURES).fillna(0.0)

    def contributions(self, X):
        dm = xgb.DMatrix(tree_frame(X, self.kind), enable_categorical=True)
        c = self.model.get_booster().predict(dm, pred_contribs=True, iteration_range=(0, self.best_iteration() + 1))
        return c[..., :-1]


class CatModel(GBMModel):
    kind = 'catboost'

    def defaults(self):
        p = dict(iterations=CFG.N_ESTIMATORS, learning_rate=CFG.LEARNING_RATE, depth=6, l2_leaf_reg=3.0,
                 random_seed=CFG.SEED, verbose=0, thread_count=-1, allow_writing_files=False,
                 early_stopping_rounds=CFG.EARLY_STOPPING)
        if IS_CLF and N_CLASSES > 10:
            p.update(max_ctr_complexity=1, boosting_type='Plain')
        if CFG.USE_GPU:
            p['task_type'] = 'GPU'
        return p

    def _fit(self, X, y, Xv, yv, w):
        params = dict(self.params)
        if IS_REG:
            params.setdefault('loss_function', 'RMSE')
            cls = CatBoostRegressor
        else:
            params.setdefault('loss_function', 'Logloss' if len(self.classes) == 2 else 'MultiClass')
            cls = CatBoostClassifier
        if Xv is None:
            params.pop('early_stopping_rounds', None)
        self.model = cls(**params)
        train_pool = Pool(X, y, cat_features=F_CAT, weight=w)
        eval_pool = Pool(Xv, yv, cat_features=F_CAT) if Xv is not None else None
        self.model.fit(train_pool, eval_set=eval_pool, use_best_model=eval_pool is not None)

    def best_iteration(self):
        return self.model.get_best_iteration() or self.params['iterations']

    def importance(self):
        return pd.Series(self.model.get_feature_importance(), index=FEATURES)

    def contributions(self, X):
        pool = Pool(tree_frame(X, self.kind), cat_features=F_CAT)
        return self.model.get_feature_importance(pool, type='ShapValues')[..., :-1]


MODEL_ZOO = {'lightgbm': LGBModel, 'xgboost': XGBModel, 'catboost': CatModel}
ACTIVE = [m for m in CFG.MODELS if m in MODEL_ZOO and AVAILABLE.get(m)]
SKIPPED = [m for m in CFG.MODELS if m not in ACTIVE]
print(f'Models to train: {ACTIVE}' + (f' | skipped (not installed or unknown): {SKIPPED}' if SKIPPED else ''))

## Cross-Validated Training

Each active model is trained on the same folds as the baseline. Fold scores, the mean best iteration (a useful sanity check: values pinned at `N_ESTIMATORS` mean the learning rate is too low, values near zero mean overfitting starts immediately), and runtime are collected.

In [ ]:
for name in ACTIVE:
    RESULTS[name] = cross_validate(name, lambda n=name: MODEL_ZOO[n]())
    RESULTS[name]['best_iter'] = float(np.mean([m.best_iteration() for m in RESULTS[name]['models']]))

if not ACTIVE:
    note('No boosting model is active, so the pipeline continues with the baseline only.')
else:
    rows = []
    for name, r in RESULTS.items():
        rows.append({'model': name, f'cv_{METRIC}': r['cv_primary'], 'cv_std': r['cv_std'],
                     f'holdout_{METRIC}': r['holdout'].get(METRIC, np.nan), 'mean_best_iter': r.get('best_iter', np.nan),
                     'seconds': r['time']})
    display(pd.DataFrame(rows).set_index('model').round(4))

    fig, ax = plt.subplots(figsize=(11, 4.6))
    for i, (name, r) in enumerate(RESULTS.items()):
        ax.plot(range(1, len(r['fold_scores']) + 1), r['fold_scores'], marker='o', lw=2, ms=6,
                color=MUTED if name == 'baseline' else PALETTE[i % 10], label=f'{name} ({r["cv_primary"]:.4f})')
    ax.set_xticks(range(1, len(FOLDS) + 1))
    ax.set_xlabel('fold')
    ax.set_ylabel(METRIC)
    ax.legend(loc='best')
    suptitle(fig, 'Per-fold scores', f'{METRIC}, {"higher" if GREATER_IS_BETTER[METRIC] else "lower"} is better; consistent gaps across folds mean a real improvement')
    save_fig(fig, 'fold_scores', 'final')

for name in ACTIVE:
    cov = RESULTS[name]['covered']
    prediction_frame(RESULTS[name]['oof'][cov], dev_raw[cov], y_dev[cov]).to_csv(CFG.FINAL_DIR / 'oof' / f'{name}.csv', index=False)

#### Insights

> *Fill in after running.* How much do the boosted models improve on the linear baseline, and is the gain consistent in every fold? Do the three libraries agree? Large disagreement hints at instability or an unhandled data issue.

## Modality Ablation

The ablation answers the question every multimodal report must answer: does the text actually help? The first active boosting model is retrained on the shared folds with the tabular features only, then with each text family added on its own, with the text features only, and with everything. A gain larger than the fold standard deviation is a real contribution. The table is saved as `modality_ablation.csv`.

In [ ]:
FAMILY = {f: origin(f) for f in FEATURES}
TEXT_FAMILIES = [fam for fam in ('text statistics', 'text TF-IDF SVD', 'text embedding', 'NLP model OOF') if fam in FAMILY.values()]
ABLATION = None
if CFG.MODALITY_ABLATION and TEXT_FAMILIES and ACTIVE:
    probe = ACTIVE[0]
    base_feats = [f for f in FEATURES if FAMILY[f] not in TEXT_FAMILIES]
    variants = {'tabular only': base_feats}
    for fam in TEXT_FAMILIES:
        variants[f'+ {fam}'] = base_feats + [f for f in FEATURES if FAMILY[f] == fam]
    variants['text only'] = [f for f in FEATURES if FAMILY[f] in TEXT_FAMILIES]
    variants['all features'] = FEATURES
    saved = (FEATURES, F_NUM, F_CAT)
    rows = []
    for label, feats in variants.items():
        if not feats:
            continue
        FEATURES, F_NUM, F_CAT = feats, [f for f in saved[1] if f in feats], [f for f in saved[2] if f in feats]
        r = cross_validate(label, lambda: MODEL_ZOO[probe](), verbose=False, eval_sets=False)
        rows.append({'variant': label, 'n_features': len(feats), f'cv_{METRIC}': r['cv_primary'], 'cv_std': r['cv_std']})
        print(f'  [{label}] {METRIC} = {r["cv_primary"]:.4f} +/- {r["cv_std"]:.4f}  ({len(feats)} features)')
    FEATURES, F_NUM, F_CAT = saved
    ABLATION = pd.DataFrame(rows).set_index('variant')
    ref = ABLATION.loc['tabular only', f'cv_{METRIC}']
    sign = 1 if GREATER_IS_BETTER[METRIC] else -1
    ABLATION['gain_vs_tabular'] = sign * (ABLATION[f'cv_{METRIC}'] - ref)
    ABLATION.to_csv(CFG.FINAL_DIR / 'modality_ablation.csv')
    display(ABLATION.round(4))

    fig, ax = plt.subplots(figsize=(11, 0.55 * len(ABLATION) + 1.6))
    colors = [MUTED if v == 'tabular only' else ACCENT if v == 'all features' else PRIMARY for v in ABLATION.index]
    ax.barh(ABLATION.index[::-1], ABLATION[f'cv_{METRIC}'][::-1], xerr=ABLATION['cv_std'][::-1], color=colors[::-1], capsize=3)
    ax.axvline(ref, color=MUTED, ls='--', lw=1)
    lo = (ABLATION[f'cv_{METRIC}'] - ABLATION['cv_std']).min()
    hi = (ABLATION[f'cv_{METRIC}'] + ABLATION['cv_std']).max()
    pad = max((hi - lo) * 0.15, 1e-3)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_xlabel(f'CV {METRIC}')
    suptitle(fig, 'Modality ablation', f'{probe} on the shared folds; dashed line = tabular only, bars = +/- fold std')
    save_fig(fig, 'modality_ablation', 'final')
else:
    print('Ablation skipped: no text features, no boosting model, or MODALITY_ABLATION = False.')

#### Insights

> *Fill in after running.* Which text family adds the most over the tabular features, and is the gain larger than the fold noise? Does text alone already beat the tabular features? Report the gain of the full model over tabular-only as the headline multimodal result.

---

# Hyperparameter Tuning <a name="9"></a>

---

When `TUNE = True`, Optuna's TPE sampler searches the hyperparameters of `TUNE_MODEL` on a lighter `TUNE_FOLDS`-fold version of the same validation scheme. The best configuration is then retrained on the full shared folds as `<model>_tuned`, so its score is directly comparable to every other model. The holdout set is never used during tuning.

In [ ]:
def suggest(trial, kind):
    lr = trial.suggest_float('learning_rate', 0.01, 0.15, log=True)
    if kind == 'lightgbm':
        return dict(learning_rate=lr, num_leaves=trial.suggest_int('num_leaves', 15, 255, log=True),
                    min_child_samples=trial.suggest_int('min_child_samples', 5, 200, log=True),
                    subsample=trial.suggest_float('subsample', 0.5, 1.0),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
                    reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30, log=True),
                    reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10, log=True))
    if kind == 'xgboost':
        return dict(learning_rate=lr, max_depth=trial.suggest_int('max_depth', 3, 10),
                    min_child_weight=trial.suggest_float('min_child_weight', 0.5, 50, log=True),
                    subsample=trial.suggest_float('subsample', 0.5, 1.0),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
                    gamma=trial.suggest_float('gamma', 1e-3, 5, log=True),
                    reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30, log=True))
    return dict(learning_rate=lr, depth=trial.suggest_int('depth', 4, 10),
                l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 30, log=True),
                random_strength=trial.suggest_float('random_strength', 1e-3, 5, log=True),
                bagging_temperature=trial.suggest_float('bagging_temperature', 0, 1))

BEST_PARAMS = None
if not CFG.TUNE:
    note('TUNE is False, so tuning is skipped. Set TUNE = True and choose TUNE_MODEL to enable it.')
elif not AVAILABLE['optuna']:
    note('optuna is not installed, so tuning is skipped.')
elif CFG.TUNE_MODEL not in ACTIVE:
    note(f'TUNE_MODEL={CFG.TUNE_MODEL!r} is not an active model ({ACTIVE}), so tuning is skipped.')
else:
    kind = CFG.TUNE_MODEL
    tune_folds = make_folds(CFG.TUNE_FOLDS)

    def objective(trial):
        params = suggest(trial, kind)
        res = cross_validate('trial', lambda: MODEL_ZOO[kind](params), folds=tune_folds, verbose=False, eval_sets=False)
        return res['cv_primary']

    study = optuna.create_study(direction='maximize' if GREATER_IS_BETTER[METRIC] else 'minimize',
                                sampler=optuna.samplers.TPESampler(seed=CFG.SEED))
    study.optimize(objective, n_trials=CFG.N_TRIALS, timeout=CFG.TUNE_TIMEOUT, show_progress_bar=False)
    BEST_PARAMS = study.best_params
    print(f'{len(study.trials)} trials | best {METRIC} = {study.best_value:.4f}')
    print('Best params:', BEST_PARAMS)
    trials = study.trials_dataframe()
    trials.to_csv(CFG.FINAL_DIR / f'optuna_trials_{kind}.csv', index=False)
    to_json(BEST_PARAMS, CFG.FINAL_DIR / f'best_params_{kind}.json')

    fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
    vals = trials['value'].values
    run_best = np.maximum.accumulate(vals) if GREATER_IS_BETTER[METRIC] else np.minimum.accumulate(vals)
    axes[0].scatter(trials['number'], vals, color=SOFT, s=25, label='trial')
    axes[0].plot(trials['number'], run_best, color=ACCENT, lw=2, label='best so far')
    axes[0].set_xlabel('trial')
    axes[0].set_ylabel(METRIC)
    axes[0].legend()
    axes[0].set_title('Optimisation history', loc='left')
    try:
        imp = pd.Series(optuna.importance.get_param_importances(study)).sort_values()
        axes[1].barh(imp.index, imp.values, color=PRIMARY)
        axes[1].set_title('Hyperparameter importance (fANOVA)', loc='left')
    except Exception:
        axes[1].set_visible(False)
    suptitle(fig, f'Optuna tuning: {kind}', f'{len(study.trials)} trials on {CFG.TUNE_FOLDS} folds')
    save_fig(fig, f'tuning_{kind}', 'final')

    RESULTS[f'{kind}_tuned'] = cross_validate(f'{kind}_tuned', lambda: MODEL_ZOO[kind](BEST_PARAMS))
    RESULTS[f'{kind}_tuned']['best_iter'] = float(np.mean([m.best_iteration() for m in RESULTS[f'{kind}_tuned']['models']]))

#### Insights

> *Fill in after running.* Did tuning improve the cross-validated score beyond fold noise (compare the gain with `cv_std`)? Which hyperparameters mattered most?

---

# Ensemble and Model Selection <a name="10"></a>

---

Different model families make different errors, so a weighted average of their predictions is usually better than any single one. Non-negative weights summing to one are fitted on the out-of-fold predictions by minimising RMSE (regression) or log loss (classification), which are smooth proxies for any selection metric. The final model is the candidate with the best cross-validated score. The holdout score is reported alongside but is not used for selection, which keeps it an honest estimate.

In [ ]:
def blend(arrs, w):
    return sum(wi * a for wi, a in zip(w, arrs))

WEIGHTS = None
cands = [k for k in RESULTS]
if CFG.ENSEMBLE and len(cands) >= 2:
    cov = np.logical_and.reduce([RESULTS[k]['covered'] for k in cands])
    oofs = [RESULTS[k]['oof'][cov] for k in cands]
    yc = y_dev[cov]

    def blend_loss(w):
        p = blend(oofs, w)
        if IS_REG:
            return np.sqrt(np.mean((yc - p) ** 2))
        return log_loss(yc, np.clip(p, 1e-7, 1), labels=list(range(N_CLASSES)))

    n = len(cands)
    opt = minimize(blend_loss, np.full(n, 1 / n), method='SLSQP', bounds=[(0, 1)] * n,
                   constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
    WEIGHTS = {k: round(float(w), 4) for k, w in zip(cands, opt.x)}
    print('Ensemble weights:', WEIGHTS)

    oof = np.full_like(RESULTS[cands[0]]['oof'], np.nan)
    oof[cov] = blend(oofs, opt.x)
    ens = {'name': 'ensemble', 'oof': oof, 'covered': cov, 'models': None, 'time': 0.0,
           'hold': blend([RESULTS[k]['hold'] for k in cands], opt.x),
           'test': blend([RESULTS[k]['test'] for k in cands], opt.x) if X_test is not None else None}
    ens['fold_scores'] = [primary(y_dev[va], oof[va]) for _, va in FOLDS if cov[va].all()]
    ens['cv'] = score_all(y_dev[cov], oof[cov])
    ens['cv_primary'], ens['cv_std'] = ens['cv'][METRIC], float(np.std(ens['fold_scores']))
    ens['holdout'] = score_all(y_hold, ens['hold']) if HAS_HOLD else {}
    RESULTS['ensemble'] = ens
    to_json(WEIGHTS, CFG.FINAL_DIR / 'ensemble_weights.json')
else:
    note('Ensembling needs ENSEMBLE = True and at least two models.')

## Leaderboard

The leaderboard lists every candidate with its cross-validated metrics and holdout score. The chart shows the CV score with its fold standard deviation and the holdout score as a marker, so overfitting to the folds (CV much better than holdout) is visible at a glance.

In [ ]:
rows = []
for name, r in RESULTS.items():
    row = {'model': name, f'cv_{METRIC}': r['cv_primary'], 'cv_std': r['cv_std'],
           f'holdout_{METRIC}': r['holdout'].get(METRIC, np.nan), 'seconds': r['time']}
    row.update({f'cv_{k}': v for k, v in r['cv'].items() if k != METRIC})
    rows.append(row)
LEADERBOARD = pd.DataFrame(rows).set_index('model').sort_values(f'cv_{METRIC}', ascending=not GREATER_IS_BETTER[METRIC])
BEST_NAME = LEADERBOARD.index[0]
BEST = RESULTS[BEST_NAME]
LEADERBOARD.to_csv(CFG.FINAL_DIR / 'leaderboard.csv')
display(LEADERBOARD.round(4))
print(f'Selected model: {BEST_NAME}  (CV {METRIC} = {BEST["cv_primary"]:.4f})')

lb = LEADERBOARD.iloc[::-1]
fig, axes = plt.subplots(1, 2 if WEIGHTS else 1, figsize=(17 if WEIGHTS else 11, 0.55 * len(lb) + 2.2), squeeze=False)
ax = axes[0, 0]
colors = [ACCENT if m == BEST_NAME else MUTED if m == 'baseline' else PRIMARY for m in lb.index]
ax.barh(lb.index, lb[f'cv_{METRIC}'], xerr=lb['cv_std'], color=colors, error_kw=dict(ecolor=DARK, lw=1, capsize=3))
if HAS_HOLD:
    ax.scatter(lb[f'holdout_{METRIC}'], lb.index, marker='D', color=DARK, zorder=3, s=40, label='holdout')
    ax.legend(loc='lower right')
vals = pd.concat([lb[f'cv_{METRIC}'] - lb['cv_std'], lb[f'cv_{METRIC}'] + lb['cv_std']] + ([lb[f'holdout_{METRIC}']] if HAS_HOLD else [])).dropna()
lo, hi = vals.min(), vals.max()
pad = max((hi - lo) * 0.25, abs(hi) * 0.02, 1e-6)
ax.set_xlim(lo - pad, hi + 2 * pad)
for i, (v, sd) in enumerate(zip(lb[f'cv_{METRIC}'], lb['cv_std'])):
    ax.text(v + sd, i, f'  {v:.4f}', fontsize=8, color='#4A5568', va='center')
ax.set_xlabel(f'{METRIC} ({"higher" if GREATER_IS_BETTER[METRIC] else "lower"} is better)')
ax.set_title('CV score +/- fold std (bars) and holdout (diamonds)', loc='left')
if WEIGHTS:
    w = pd.Series(WEIGHTS).sort_values()
    axes[0, 1].barh(w.index, w.values, color=PALETTE[2])
    bar_labels(axes[0, 1], '{:.2f}')
    axes[0, 1].set_title('Ensemble weights', loc='left')
suptitle(fig, 'Model leaderboard', f'selected: {BEST_NAME} (orange), baseline in grey')
save_fig(fig, 'leaderboard', 'final')

#### Insights

> *Fill in after running.* Which model was selected and by how much does it beat the baseline in CV and holdout? Do CV and holdout agree? A holdout much worse than CV signals overfitting or a distribution difference. Which models carry most of the ensemble weight?

---

# Final Evaluation on Holdout <a name="11"></a>

---

The selected model is now scored once on the holdout set, which played no part in training, tuning, weighting, or selection. For binary tasks scored by a threshold-dependent metric (F1, accuracy), the decision threshold is first tuned on out-of-fold predictions. A bootstrap over holdout rows gives a 95% confidence interval for the score and for the improvement over the baseline, which is the evidence the report needs to claim the improvement is real.

In [ ]:
if IS_BINARY and METRIC in ('f1', 'f1_macro', 'f1_weighted', 'accuracy', 'balanced_accuracy'):
    cov = BEST['covered']
    grid = np.linspace(0.05, 0.95, 91)
    scores = []
    for t in grid:
        THRESHOLD[0] = t
        scores.append(primary(y_dev[cov], BEST['oof'][cov]))
    THRESHOLD[0] = float(grid[int(np.argmax(scores))])
    print(f'Tuned decision threshold on OOF: {THRESHOLD[0]:.2f} ({METRIC} {max(scores):.4f} vs {scores[45]:.4f} at 0.50)')

if HAS_HOLD:
    EVAL_NAME, EVAL_Y = 'holdout', y_hold
    EVAL_P = {k: RESULTS[k]['hold'] for k in RESULTS}
    EVAL_RAW = hold_raw
else:
    note('HOLDOUT_SIZE is 0, so the final evaluation uses out-of-fold predictions instead.')
    cov = np.logical_and.reduce([r['covered'] for r in RESULTS.values()])
    EVAL_NAME, EVAL_Y = 'out-of-fold', y_dev[cov]
    EVAL_P = {k: RESULTS[k]['oof'][cov] for k in RESULTS}
    EVAL_RAW = dev_raw[cov].reset_index(drop=True)

naive_eval = naive_prediction(len(EVAL_Y))
final_table = pd.DataFrame({'naive': score_all(EVAL_Y, naive_eval),
                            'baseline': score_all(EVAL_Y, EVAL_P['baseline']),
                            BEST_NAME: score_all(EVAL_Y, EVAL_P[BEST_NAME])})
sign = 1 if GREATER_IS_BETTER[METRIC] else -1

rng = np.random.RandomState(CFG.SEED)
boot_best, boot_diff = [], []
for _ in range(200):
    b = rng.randint(0, len(EVAL_Y), len(EVAL_Y))
    sb = primary(EVAL_Y[b], EVAL_P[BEST_NAME][b])
    boot_best.append(sb)
    boot_diff.append(sign * (sb - primary(EVAL_Y[b], EVAL_P['baseline'][b])))
ci = np.nanpercentile(boot_best, [2.5, 97.5])
ci_diff = np.nanpercentile(boot_diff, [2.5, 97.5])
FINAL = {'model': BEST_NAME, 'evaluated_on': EVAL_NAME, 'metric': METRIC, 'threshold': THRESHOLD[0],
         'scores': final_table[BEST_NAME].to_dict(), 'baseline_scores': final_table['baseline'].to_dict(),
         'naive_scores': final_table['naive'].to_dict(), 'ci95': ci.tolist(), 'improvement_vs_baseline_ci95': ci_diff.tolist()}
display(final_table.round(4))
rel = sign * (final_table.loc[METRIC, BEST_NAME] - final_table.loc[METRIC, 'baseline']) / abs(final_table.loc[METRIC, 'baseline']) * 100
print(f'{BEST_NAME} {EVAL_NAME} {METRIC} = {final_table.loc[METRIC, BEST_NAME]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f})')
print(f'Improvement over baseline: {rel:+.1f}% (bootstrap 95% CI of the gain: {ci_diff[0]:+.4f} to {ci_diff[1]:+.4f}, '
      f'{"significant" if ci_diff[0] > 0 else "not significant"})')
final_table.to_csv(CFG.FINAL_DIR / 'final_metrics.csv')
to_json(FINAL, CFG.FINAL_DIR / 'final_metrics.json')

## Final Diagnostics

The diagnostic panels mirror the baseline ones, so the two can be placed side by side in the report. For binary classification a calibration curve is added, because well-calibrated probabilities are what make a score usable as a business risk estimate.

In [ ]:
plot_diagnostics(EVAL_Y, EVAL_P[BEST_NAME], f'Final model ({BEST_NAME}) {EVAL_NAME} diagnostics', 'final',
                 f'{METRIC} = {final_table.loc[METRIC, BEST_NAME]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f}) vs baseline {final_table.loc[METRIC, "baseline"]:.4f}')

if IS_BINARY:
    fig, ax = plt.subplots(figsize=(7.5, 6))
    for name, col in ((BEST_NAME, ACCENT), ('baseline', MUTED)):
        frac, mean_pred = calibration_curve(EVAL_Y, EVAL_P[name][:, 1], n_bins=10, strategy='quantile')
        ax.plot(mean_pred, frac, marker='o', color=col, lw=2, label=name)
    ax.plot([0, 1], [0, 1], color=DARK, ls='--', lw=1, label='perfect')
    ax.set_xlabel('mean predicted probability')
    ax.set_ylabel('observed frequency')
    ax.legend()
    suptitle(fig, 'Calibration curve', f'{EVAL_NAME} set, quantile bins')
    save_fig(fig, 'calibration', 'final')

#### Insights

> *Fill in after running.* Quote the final score with its confidence interval and the improvement over the baseline. Is the gain significant (the CI of the gain excludes zero)? Where do the diagnostics show systematic error, for example under-prediction of high values?

---

# Model Interpretation <a name="12"></a>

---

Interpretation uses the best tree model (the ensemble itself is a weighted average of models, so its strongest tree component is explained). Gain importance shows how much each feature reduces loss inside the trees. Exact TreeSHAP contributions, computed with each library's own API and averaged over the fold models, show both how much and in which direction each feature moves individual predictions.

In [ ]:
TREE_NAMES = [k for k in RESULTS if RESULTS[k].get('models') and isinstance(RESULTS[k]['models'][0], GBMModel)]
INTERP = min(TREE_NAMES, key=lambda k: -sign * RESULTS[k]['cv_primary']) if TREE_NAMES else None
IMPORTANCE = pd.DataFrame()
if INTERP is None:
    note('No tree model was trained, so interpretation relies on the baseline coefficients in section 7.')
else:
    models = RESULTS[INTERP]['models']
    gain = pd.concat([m.importance() / max(m.importance().sum(), 1e-12) for m in models], axis=1).mean(axis=1)
    X_int = (X_hold if HAS_HOLD else X_dev).sample(min(2000, len(X_hold) if HAS_HOLD else len(X_dev)), random_state=CFG.SEED)
    contribs = [m.contributions(X_int) for m in models]
    mean_abs = np.mean([np.abs(c).mean(axis=tuple(range(c.ndim - 1))) for c in contribs], axis=0)
    IMPORTANCE = pd.DataFrame({'gain_share': gain, 'mean_abs_shap': mean_abs}, index=FEATURES)
    IMPORTANCE = IMPORTANCE.sort_values('mean_abs_shap', ascending=False)
    IMPORTANCE.to_csv(CFG.FINAL_DIR / 'feature_importance.csv')
    display(IMPORTANCE.head(15).round(4))

    top = IMPORTANCE.head(20).iloc[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(17, 0.34 * len(top) + 2), sharey=True)
    cols = [ACCENT if c in F_CAT else PRIMARY for c in top.index]
    axes[0].barh([short(c, 30) for c in top.index], top['mean_abs_shap'], color=cols)
    axes[0].set_title('Mean |SHAP| (impact on prediction)', loc='left')
    axes[1].barh([short(c, 30) for c in top.index], top['gain_share'], color=cols)
    axes[1].set_title('Gain share inside the trees', loc='left')
    suptitle(fig, f'Feature importance: {INTERP}', 'orange = categorical, blue = numeric; averaged over fold models')
    save_fig(fig, 'feature_importance', 'final')

## SHAP Summary and Dependence

For regression and binary classification, the beeswarm shows each sampled row as a dot: its horizontal position is the feature's contribution and its colour is the feature value (red is high, blue is low, grey is categorical). The dependence panels then plot contribution against value for the four strongest numeric features, which turns the model into business statements such as 'popularity rises with danceability up to 0.7 and then flattens'.

In [ ]:
if INTERP is not None and (IS_REG or IS_BINARY):
    S = np.mean(contribs, axis=0)
    feats = IMPORTANCE.index[:15].tolist()
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(feats) + 2))
    cmap = LinearSegmentedColormap.from_list('bw', ['#3D5A80', '#D6E4F0', '#EE6C4D'])
    jitter = np.random.RandomState(CFG.SEED)
    for i, f in enumerate(feats[::-1]):
        j = FEATURES.index(f)
        sv = S[:, j]
        yj = i + jitter.uniform(-0.3, 0.3, len(sv))
        if f in F_CAT:
            ax.scatter(sv, yj, s=6, color=MUTED, alpha=0.5)
        else:
            v = X_int[f].rank(pct=True).values
            ax.scatter(sv, yj, s=6, c=v, cmap=cmap, alpha=0.7)
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels([short(f, 30) for f in feats[::-1]])
    ax.axvline(0, color=DARK, lw=1)
    scale = f'log1p({T})' if (IS_REG and TRANSFORM) else (T if IS_REG else 'log-odds')
    ax.set_xlabel(f'SHAP value (contribution to {scale})')
    sm = plt.cm.ScalarMappable(cmap=cmap)
    cb = fig.colorbar(sm, ax=ax, shrink=0.6, ticks=[0, 1])
    cb.ax.set_yticklabels(['low', 'high'])
    suptitle(fig, f'SHAP summary: {INTERP}', f'{len(X_int):,} sampled {EVAL_NAME} rows, grey = categorical feature')
    save_fig(fig, 'shap_beeswarm', 'final')

    dep = [f for f in IMPORTANCE.index if f in F_NUM][:4]
    if dep:
        fig, axes = plt.subplots(1, len(dep), figsize=(4.6 * len(dep), 4.2), squeeze=False)
        for ax, f in zip(axes[0], dep):
            x = X_int[f].values
            sv = S[:, FEATURES.index(f)]
            ok = ~np.isnan(x)
            lo, hi = np.nanpercentile(x, [1, 99])
            m = ok & (x >= lo) & (x <= hi)
            ax.scatter(x[m], sv[m], s=7, color=PRIMARY, alpha=0.35)
            d = pd.DataFrame({'x': x[m], 's': sv[m]})
            b = d.groupby(pd.qcut(d['x'].rank(method='first'), 15), observed=True).mean()
            ax.plot(b['x'], b['s'], color=ACCENT, lw=2.2)
            ax.axhline(0, color=DARK, lw=0.8)
            ax.set_title(short(f, 26), loc='left', fontsize=10)
            ax.set_ylabel('SHAP value')
        suptitle(fig, 'SHAP dependence of the top numeric features', 'orange = binned mean contribution')
        save_fig(fig, 'shap_dependence', 'final')
elif INTERP is not None:
    note('Multiclass SHAP has one contribution per class, so only the mean |SHAP| ranking above is shown.')

#### Insights

> *Fill in after running.* Which features drive predictions, and in which direction? Do the SHAP directions agree with the baseline coefficients and the EDA? Translate the top three dependence shapes into plain business statements for the presentation.

---

# Error Analysis <a name="13"></a>

---

An aggregate score hides where the model fails. This section breaks the evaluation-set error down by the most important categorical feature and by the size of the target (regression), or by class and confusion pair (classification), and lists the worst individual predictions. These are the cases to inspect manually and the limitations to state honestly in the report.

In [ ]:
pred_best = EVAL_P[BEST_NAME]
seg_cols = [c for c in (IMPORTANCE.index if not IMPORTANCE.empty else F_CAT) if c in F_CAT]
seg_col = next((c for c in seg_cols if EVAL_RAW[c].nunique() <= 60), seg_cols[0] if seg_cols else None)

if IS_REG:
    err = pd.DataFrame({'actual': EVAL_Y, 'pred': pred_best})
    err['residual'] = err['actual'] - err['pred']
    err['abs_error'] = err['residual'].abs()
    err['decile'] = pd.qcut(err['actual'].rank(method='first'), 10, labels=[f'D{i}' for i in range(1, 11)])
    by_dec = err.groupby('decile', observed=True).agg(mae=('abs_error', 'mean'), bias=('residual', 'mean'), actual_mean=('actual', 'mean'))
    ncols = 2 if seg_col else 1
    fig, axes = plt.subplots(1, ncols, figsize=(8.5 * ncols, 5.5), squeeze=False)
    ax = axes[0, 0]
    ax.bar(by_dec.index.astype(str), by_dec['mae'], color=SOFT, label='MAE')
    ax.plot(by_dec.index.astype(str), by_dec['bias'], color=ACCENT, marker='o', lw=2, label='bias (actual - pred)')
    ax.axhline(0, color=DARK, lw=0.8)
    ax.set_xlabel(f'decile of actual {T} (D1 = lowest)')
    ax.legend()
    ax.set_title('Error by target size', loc='left')
    if seg_col:
        err['segment'] = EVAL_RAW[seg_col].astype(str).values
        top_lv = err['segment'].value_counts().index[:15]
        by_seg = err[err['segment'].isin(top_lv)].groupby('segment').agg(mae=('abs_error', 'mean'), bias=('residual', 'mean'), n=('abs_error', 'size')).sort_values('mae')
        ax = axes[0, 1]
        ax.barh([short(i, 22) for i in by_seg.index], by_seg['mae'], color=[ACCENT if b > 0 else PRIMARY for b in by_seg['bias']])
        ax.axvline(err['abs_error'].mean(), color=DARK, ls='--', lw=1)
        ax.set_title(f'MAE by {seg_col} (orange = under-predicted on average)', loc='left')
        by_seg.to_csv(CFG.FINAL_DIR / f'error_by_{slug(seg_col)}.csv')
    suptitle(fig, 'Where does the final model fail?', f'{EVAL_NAME} set')
    save_fig(fig, 'error_analysis', 'final')
    by_dec.to_csv(CFG.FINAL_DIR / 'error_by_target_decile.csv')
    worst = EVAL_RAW.assign(actual=EVAL_Y, predicted=pred_best, abs_error=np.abs(EVAL_Y - pred_best)).nlargest(15, 'abs_error')
else:
    lab = to_label(pred_best)
    p, r, f, s = precision_recall_fscore_support(EVAL_Y, lab, labels=list(range(N_CLASSES)), zero_division=0)
    report = pd.DataFrame({'precision': p, 'recall': r, 'f1': f, 'support': s}, index=CLASS_NAMES).sort_values('f1')
    report.to_csv(CFG.FINAL_DIR / 'classification_report.csv')
    display(report.round(3).head(15))
    cm = pd.crosstab(pd.Series(EVAL_Y, name='actual'), pd.Series(lab, name='predicted'))
    pairs = cm.stack().rename('count').reset_index()
    pairs = pairs[pairs['actual'] != pairs['predicted']].nlargest(12, 'count')
    pairs['pair'] = [f'{short(CLASS_NAMES[a], 14)} -> {short(CLASS_NAMES[b], 14)}' for a, b in zip(pairs['actual'], pairs['predicted'])]
    ncols = 2 if seg_col else 1
    fig, axes = plt.subplots(1, ncols, figsize=(8.5 * ncols, 5.5), squeeze=False)
    axes[0, 0].barh(pairs['pair'][::-1], pairs['count'][::-1], color=ACCENT)
    axes[0, 0].set_title('Most frequent confusions (actual -> predicted)', loc='left')
    if seg_col:
        seg = pd.Series(EVAL_RAW[seg_col].astype(str).values)
        top_lv = seg.value_counts().index[:15]
        er = pd.Series(lab != EVAL_Y).groupby(seg).mean().reindex(top_lv).sort_values()
        axes[0, 1].barh([short(i, 22) for i in er.index], er.values, color=PRIMARY)
        axes[0, 1].axvline((lab != EVAL_Y).mean(), color=DARK, ls='--', lw=1)
        axes[0, 1].set_title(f'Error rate by {seg_col}', loc='left')
    suptitle(fig, 'Where does the final model fail?', f'{EVAL_NAME} set')
    save_fig(fig, 'error_analysis', 'final')
    conf = pred_best.max(axis=1)
    wrong = EVAL_RAW.assign(actual=LE.inverse_transform(EVAL_Y), predicted=LE.inverse_transform(lab), confidence=conf)[lab != EVAL_Y]
    worst = wrong.nlargest(15, 'confidence')

show_cols = [c for c in worst.columns if c not in TEXT_COLS][:12]
worst[show_cols].to_csv(CFG.FINAL_DIR / 'worst_predictions.csv', index=False)
display(worst[show_cols].head(10))

#### Insights

> *Fill in after running.* Which segments or classes are hardest? Does the model shrink extreme targets toward the mean (positive bias in the top deciles, negative in the bottom)? Inspect the worst rows: are they label noise, missing information, or a fixable feature gap?

---

# Predictions and Export <a name="14"></a>

---

The final artefacts are written to `pipeline-output/final/`: predictions for the out-of-fold, holdout, and test sets (with `submission.csv` when a test file is configured), the fold models together with the fitted preprocessing objects, the resolved configuration, and a markdown report that summarises the whole run.

## Inference Function

`predict_new` reproduces the full path from raw rows to predictions with the fitted feature builder and the selected model (or ensemble). It is verified against the stored holdout predictions, which guarantees that the exported submission uses exactly the evaluated model.

In [ ]:
def model_predict(name, X):
    if name == 'ensemble':
        parts = [model_predict(k, X) for k in WEIGHTS]
        return blend(parts, list(WEIGHTS.values()))
    return np.mean([m.predict(X) for m in RESULTS[name]['models']], axis=0)

def predict_new(raw):
    X = FB.transform(prepare(raw)).reset_index(drop=True)
    X[F_NUM] = X[F_NUM].astype(float)
    return model_predict(BEST_NAME, X)

if HAS_HOLD:
    check = predict_new(hold_raw.head(200))
    ok = np.allclose(check, RESULTS[BEST_NAME]['hold'][:200], atol=1e-4)
    print(f'predict_new reproduces stored holdout predictions: {ok}')
display(prediction_frame(predict_new(dev_raw.head(5)), dev_raw.head(5)))

## Export Artefacts

Saved models are pickles of the notebook's wrapper classes, so they are reloaded inside this notebook (or any script that defines the same classes) with `joblib.load`.

In [ ]:
export_result(BEST, CFG.FINAL_DIR)

names = list(WEIGHTS) if BEST_NAME == 'ensemble' else [BEST_NAME]
for k in names:
    joblib.dump(RESULTS[k]['models'], CFG.FINAL_DIR / 'models' / f'{k}_fold_models.joblib')
joblib.dump({'feature_builder': FB, 'cat_levels': CAT_LEVELS, 'label_encoder': LE if IS_CLF else None,
             'features': FEATURES, 'num_features': F_NUM, 'cat_features': F_CAT, 'transform': TRANSFORM,
             'threshold': THRESHOLD[0], 'best': BEST_NAME, 'weights': WEIGHTS},
            CFG.FINAL_DIR / 'models' / 'preprocessing.joblib')

settings = {k: v for k, v in vars(Settings).items() if not k.startswith('__') and not callable(v)}
resolved = {'task': TASK, 'metric': METRIC, 'scheme': SCHEME, 'target_transform': TRANSFORM, 'class_weights': USE_CW,
            'n_classes': N_CLASSES, 'n_dev': len(X_dev), 'n_holdout': len(X_hold), 'n_test': 0 if X_test is None else len(X_test),
            'features': FEATURES, 'active_models': ACTIVE, 'skipped_models': SKIPPED, 'best_model': BEST_NAME,
            'ensemble_weights': WEIGHTS, 'tuned_params': BEST_PARAMS, 'threshold': THRESHOLD[0], 'libraries': AVAILABLE}
decision_table = pd.DataFrame(DECISIONS).set_index('setting')
decision_table.to_csv(CFG.FINAL_DIR / 'decisions.csv')
to_json({'settings': settings, 'resolved': resolved, 'eda_decisions': EDA}, CFG.FINAL_DIR / 'config.json')

## Run Report

The report collects the key numbers of the run into `pipeline-output/final/report.md`. It is a factual starting point for the methodology and results chapters of the written report.

In [ ]:
fmt = lambda v: f'{v:.4f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else '-'
lb_cols = [f'cv_{METRIC}', 'cv_std', f'holdout_{METRIC}']
lines = [f'# Modelling Report: {DATA_PATH.name}', '',
         '## Setup',
         f'- Task: **{TASK}**' + (f' ({N_CLASSES} classes)' if IS_CLF else '') + f', target `{T}`, metric **{METRIC}**',
         f'- Rows: {len(X_dev):,} development, {len(X_hold):,} holdout' + (f', {len(X_test):,} test' if X_test is not None else ''),
         f'- Validation: {SCHEME} {CFG.N_FOLDS}-fold' + (f' grouped by `{CV_GROUP}`' if SCHEME == 'group' else ''),
         f'- Target transform: {TRANSFORM}; class weights: {USE_CW}',
         f'- Features: {len(FEATURES)} ({len(F_NUM)} numeric, {len(F_CAT)} categorical)', '',
         '## Decisions', '', '| setting | value | source | why |', '|---|---|---|---|']
lines += [f'| {k} | `{r.value}` | {r.source} | {r.why} |' for k, r in decision_table.iterrows()]
lines += ['', '## Leaderboard', '', '| model | ' + ' | '.join(lb_cols) + ' |', '|---' * (len(lb_cols) + 1) + '|']
for m, r in LEADERBOARD.iterrows():
    lines.append(f'| {m}{" (selected)" if m == BEST_NAME else ""} | ' + ' | '.join(fmt(r[c]) for c in lb_cols) + ' |')
lines += ['', f'## Final result ({EVAL_NAME})',
          f'- {BEST_NAME}: {METRIC} = {FINAL["scores"][METRIC]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f})',
          f'- Baseline: {FINAL["baseline_scores"][METRIC]:.4f}; naive: {FINAL["naive_scores"][METRIC]:.4f}',
          f'- Improvement over baseline: {rel:+.1f}% (gain CI {ci_diff[0]:+.4f} to {ci_diff[1]:+.4f})',
          '- All metrics: ' + ', '.join(f'{k} {v:.4f}' for k, v in FINAL['scores'].items())]
if WEIGHTS:
    lines.append('- Ensemble weights: ' + ', '.join(f'{k} {v:.2f}' for k, v in WEIGHTS.items()))
if not IMPORTANCE.empty:
    lines += ['', f'## Top features ({INTERP}, mean |SHAP|)'] + [f'{i + 1}. `{f}` ({v:.4f})' for i, (f, v) in enumerate(IMPORTANCE['mean_abs_shap'].head(10).items())]
top_coef = coef_table.head(5)
lines += ['', '## Baseline top coefficients'] + [f'- `{f}`: {v:+.4f}' for f, v in top_coef['coefficient'].items()]
report_md = '\n'.join(lines)
(CFG.FINAL_DIR / 'report.md').write_text(report_md, encoding='utf-8')
display(Markdown(report_md))

The last cell prints the output tree, so every artefact can be located quickly when writing the report or uploading the submission.

In [ ]:
for stage in (CFG.BASELINE_DIR, CFG.FINAL_DIR):
    print(f'{stage}/')
    for p in sorted(stage.rglob('*')):
        if p.is_file():
            print(f'   {str(p.relative_to(stage)):<55} {p.stat().st_size / 1e3:>9,.1f} KB')

#### Insights

> *Fill in after running.* Summarise the final model, its honest score with confidence interval, the gain over the baseline, the top drivers, and the main limitation found in the error analysis. These five points form the backbone of the conclusion chapter.